# BSCAN — Complete pipeline: smoke test, all training runs incl. the trained ablation, and all analyses

**Code version `f6b8ae4772ce`** (identical in every notebook of this set; generated by `.scripts/build_colab_notebook.py`).

**How to run:** Runtime → Change runtime type → **GPU** (T4 or better) → Runtime → **Run all**. Allow Google Drive access when asked.
Results are saved to `MyDrive/BSCAN_f6b8ae4772ce/` after every run. If Colab disconnects, simply **Run all again**: finished runs are skipped.
Each code version uses its own Drive folder, so results of different code versions never mix.
At the end a zip file is downloaded — **send it back** (or copy it into the project and run `python .scripts/build_all.py --import <zip>`).

Stages in this notebook: B, C, D, E + optional SSL comparator.
The data are downloaded from their original hosts and every file is checked against its SHA-256. The CUDA smoke test must pass
before anything is trained. Please do not edit code cells; send any error output instead.

In [ ]:
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/BSCAN_f6b8ae4772ce"   # one folder per code version: no mixing
STAGES = ['B', 'C', 'D', 'E']          # B, C, D: training stages from configs/experiments.yaml; E: analyses
RUN_OPTIONAL_SSL = True
CODE_VERSION = 'f6b8ae4772ce'
WORK = "/content/bscan"

In [ ]:
import subprocess, sys, os, json, time, shutil, hashlib, glob
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
import torch
assert torch.cuda.is_available(), "No CUDA GPU: Runtime -> Change runtime type -> GPU, then Run all."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
TIMING = {}
def tick(k): TIMING[k] = time.time()
def tock(k): TIMING[k] = round(time.time() - TIMING[k], 1); print(f"[time] {k}: {TIMING[k]} s")

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)

In [ ]:
tick("pip")
pkgs = ["librosa==1.0.0", "soundfile==0.14.0", "pyyaml", "tabulate", "ultralytics-thop"]
if RUN_OPTIONAL_SSL: pkgs.append("transformers")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
tock("pip")
import importlib
for m in ["numpy", "scipy", "sklearn", "librosa", "soundfile", "pandas", "yaml"]:
    print(m, getattr(importlib.import_module(m), "__version__", "?"))

In [ ]:
import base64, gzip
FILES = {"configs/base.yaml": "H4sIAAAAAAAC/1VTy24bMQy8+ysI9OK0fqZJCmxPQdGe+jCaAj0UwYKW6F3BWnFLaeO6X19y7Tx68Zojajgakq/grkUhD552OMSSYccC9EByBPrTk4SOUoHpA8aBMnRBRM9LS8ASmpAwggwJDi0JQSgQMuwTH1I1eQW3HrufsKb5FSzHzwy2WFwL65sZrK+BenZtnkGPJVByBNf6P2IhHGC1uF5ezqAR9HMXQw/rxWoGt182Fwtl/tBialTOTrj7X0ziQlvmPUwde1puNS/ibt+jlEXoj2l7YcoA5uBacvueg74uUyRXAqcZfP4OWU/8EENqAJMHQolHyIX73qAhE6gZwaNdgM/f7u5GQoDpf0I0z+sr3r3u0AnPP63hjYZvX6Nzg6A7ztSzoF5kLBoXTcZij7w4yzOy0grllqM3Ww14Ufjjx+9wUq+B9mku5Fi8Scz6R82ZqheA8EurqnUqZXWvfgb/WCGxdMqXT3y56DeX4DLoPBijOoQpUYTpeOa0iZ5iwcfvfPy9GE3KRJ78mdiCEwv2fQz6tMKwOZbWDP46dJvjeGdz/MHi2vfKVki6kMbysCfRohmEfuvEqTETHkrtg1QKZRvRpU5cnqgPWE0AeuHCjmMFm7WGHRW0o9rlh+opWg4p7FRJ/Qgs9FizcfCBa2EuFRiskEPt/6neeHOMFbex6KgCx6kIx6jCAA4heT7UmVwFN4uVIqyrE7Gv1PC1xWno6gOLvipXcDnpdCqjyU5oZNvsMGm0FUzKrym/OooziDvn7hVvg/eUVE2n/FcK6FhptQqKDHQOC3U9C8YaS0lPJ164V+NMx9tJEQzJqlprKri6tJK2inUOf5VtfaNAFEterVZm44FC06rt5PD4Aj6trF64NufPe1vBGJ02t96hKzxSvUSfc622rXVta13ZyFsfuv5Z+ctxOKOkk2/6nzailiEqGZHY7aYRarSQNRxTHbkJxTyut2NnL1X95B/XxHzV6gQAAA==", "configs/bscan_controlled.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuSB+cXJiXnxyfl5JUX5OTmoKFwCqwafsJwAAAA==", "configs/bscan_orig6s.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuSB+cXJiXnx+UWa6WTFXSmJJohWXgkJxckYqSBIqDADScHvqOgAAAA==", "configs/dual_no_se.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+SmliTnxefnxxKldufkpqjhWXgkJpcWo8SG1aYg5QGAB9bDVwOAAAAA==", "configs/dual_no_tattn.yaml": "H4sIAAAAAAAC/xXJuw3AMAgA0Z4pmCADsAzCMqn4RAYX2T5Oc9LpDSklHKfXK24Q4ufnFuNIbukO8JxqBIi7lFv9yXX4J8JbrBQ+YVcPhEYAAAA=", "configs/dual_plain.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+SmliTnxBTmJmHldufkpqjhWXgkJpcWo8SG1aYk5xKpRfkppbkF8EVJtYUpIHkwIACCHqEVMAAAA=", "configs/experiments.yaml": "H4sIAAAAAAAC/71Y328aORB+568YpS9wXUigVR6o7qSEpHcPIY0aVTpdVa3MYsCq196zTRoux/9+M/Yuu0sILKl0PCTg9edvZjy/dt7A9WPGjUi5cmD4XFhnVkNwCw5WqLnEf3ppEg56Bs4s3QJm2gB/4GYFZqkQkmnj+BSE8qCM4Wm91hu41Y5PtP5uEcaEOuUPTC6Z48AfWeLkinZbTmfYD+DYRPLTmZgvDT8NR8KcK26Y08YiCZvWcUgQkD2bGJE5i2DFZPx30stWMGNCWhAzYAql5H8vhUEJSVxhIRWWNANUA38xyCRL+ELLqZc7HAxTRCTIvQLFUg6/wlOiFcq3juOnzGinEy3xu+V8+kR/1gBtw+1SoiQk2Gmv1+vgWd+Fmg5B8aVhEtreEiiJVjDSkk068C9Yiw9mRv/DFVguZ127RAs+CIv7uEo0igVvQSKOGUDqCY9KsNQJnTu6+wLBDh2vwRVPhBVakeUSbaaBcnA2OO/2z7pn7yKYcLxG7u2TC4fmuL+/gaAEGhsdgU+HeBjAL/5m/7y5734OEuSGM7w75YlkZFyLRGrK0GCJTjPmLw6Pn9LWjY/8WKADoOuAcDZnygmCQwWP65JB4a4PD8wIhm7ZHn8cjbpayVUEYy7f0k9/NonlFoajIhPJHGncASatRudBBjrHQn/wzm8m9XO2y/vRxS1JFmQlyQR6du7VPgg4rVZUaUtcRLNJ/aPzAf7QMsWnhpwEWT0cA4CzZOEpAAongUdwHJGWO3TLVMhVq+XlilP0hSF8fT/4li+gmThzuIQSR17cb61Wig4g7bAFMLEJUzFa2RktJd4NPKEXcDmEk6BPm2xzg7bpnESQfwLlkMTJNLpU8WCGTBhrFsEFCkFVb11vKLUR83PcSZ86ZQT0jAKPnAE5Eu5jC48qiSvrUUPelMuYrntYqFHyIsp7AkwMU8mi1LTUlRgxRSjnPWKbs0Bsc8pZktRJN5w3hfs9J23Ameu4izN9mXP8U5zjPZxoW+LdbdsQW+3pksnO8bZ9O959n3RcrDSmy+E25xU+ikBpuL+u8lU5i9g+zncLTpTUqeFuTsdTzEzovbgHdUKOiufWaI/gxHoi1D49fTbCb4dIm3HKRKlYVi+09NvR7W3XuhUW8fYLeWHCsOZgbdlh2z1+62Nlnqa1bLQdML+Px7UbfQ2lr2/EaPEabbyYT+qUG8bPvs5RWaedmKpFYqFNe9HEXTZXmpY6FUvbBdakZOniUp6KJP0+THkoqdiAbMy+Q55qdtwvT4Ns+ZNCWRk/SmtiX6RrNxKKd95ndLYaiooAEhexQ4n3xDl2CRtqZFy3sOKhknMO3d/AV0lsFdu+bkWbMhiFYtxpUXNE1eySxHtTq/iZyLzOgGVYzETiqVtk1i48+fOGz4pgyTDEjiFnCVV1vRMariuq1eid0FEQsG4IVHQ5xS6kaOTKcLXYJnY3Rb+4PiQy1FVgoxs6yucaFbWuCIfG+mwKVnQkMH01MC8dR4ta5v/odUCfxKNjgT4TH8m4SafHGmevZw4O2vUFFxi81gUGr9VyP7CeYvZFn4+ePJ10t6LI8Bn21wrfKtsqf030Lx8Jl7KDjFd5cqCsQF2xE763xzQkHnHp8vrjp8/X9JgSbHjHo9YZiXKIf7N4XeYom+/1qyL1ZXyzgN2Db+SaL+ObeugeC/ykBs0SyMv4ZnnkMH5/OjmAP5hV6vibT6OLmyG5c/mWjq3JZm6BBTEO3URRNHoZ2rf2nFqtytPOdlDuaI6OKIwH4YOG8Ma19Sh4gxRYb0WjY4voPvg2OzY7QuXZK7yfMWzoVpZbGrEUfUGy4Mn3TAtFV+3jrof5cmkdtgTYBBUQaoTK9aGXzTdI8PVkO1rj+K4fBk7vB9h/nWyyybMHRZqoPfjmD7eZFI6OR8m4oYEZDSci4I/5z8uP3fGnvyoL4+vbAE3ZYzwTklrA87OzFoTWcafQ4SK35Tqo0P8iN95fxvw4cNOYYz8eBj3C0sisTa0ceEi4e3g/+ABX/EZjsaHh58WXUQTj5Jan2EDTwrQYt3Vo8HnhCWi6RDKGySgr50HeZ8QMJtotvN2AGV4dOrkF/ns+YqKRW4ZmyIdV5bTp5UkTtPvvaqoh6K7vXz/P/fcBClx5TneJocF2lYpJpX6t9+8r6uShbUccR3sPbCtrQ8ONvgg02esT/oGNG+ddN7BhPd8cQOzKzo0hIRIPbK8G7aGTa+3fuvUfZ6Zv3T4YAAA=", "configs/lcnn_lfcc.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+TnJeXnxOWnIyV25+SmqOFZeCAkICyEkqSsxLzkgttlKIBqmK5QIAEfC3UkcAAAA=", "configs/lfcc_only.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+Tlpycnx+Xk4lV25+SmqOFZeCQlJRYl5yRmqxlUI0SDaWCwBQCq63OgAAAA==", "configs/mel_mfcc.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+bmpOfG5acjJXbn5Kao4Vl4JCUlFiXnJGarGVQjRQUkcBJBvLBQDnF+SNPgAAAA==", "configs/mel_only.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+bmpOfH5eTiVXbn5Kao4Vl4JCUlFiXnJGarGVQjRQMpYLANLs6Hk4AAAA", "configs/mfcc_only.yaml": "H4sIAAAAAAAC/0tKLE61UkgCknqVibk5XHmJuUB+blpycnx+Xk4lV25+SmqOFZeCQlJRYl5yRmqxlUI0SDaWCwBgDYLpOgAAAA==", "src/bscan/__init__.py": "H4sIAAAAAAAC/w3IOwqAMBAFwD6nWLZPEEvBQu1tPEBYzFMWQwzxd33TDcPM4zINc0cRcsgOK58U0Ii0S1SSJ+hJAcibHKi4sd56JsqaETXBMbMx3r8oV33vqSduXeMaG/Cy+QHkvI/XYQAAAA==", "src/bscan/analysis.py": "H4sIAAAAAAAC/51Y33PbuBF+11+B8qElpxRj5y59cKvMXHvntziei286U42GA5EgxYYEGYCUo/P5f++3C5IiJcvJVGNLxAJYfPt7Qc/zPh1sqyrZFolQxtRGSC3Lgy0sHlKRyLLYGszWWtiuqqQplBX+/U5a/F5f86Lrd0G0WPwy340/O7IOha5bzKkkrVtZ3gi1V+YgjEpqkxY6F3UmpLBNWbS0UVpb5Fqloq3Fw714Ix7uxJvFLT3d3gnZinanhOn0X6zYA2DK+JapMsWeNu2Msru6TENG56SCDAAsjRKt3HYlRqmoAWLRGLU0Ki8A1YCWm7prgMjeLAQ+wNd0NgQ0JT8rI/zPun7Uw9AGociVTmniA/2W6gDS8CisakNmk3a9DnFIW5QKHD/d/bpsTP31MKG1RhYlDl9ajHWixLbQICdl0RAm4Ut9gA50rVUQ9vh0VqS81v/DAq0Sy6MC/gjEl67QzD1a3EINSia7UcaQ9ZjJ0qqlVjkQ7hUrimyQ1FXTsZZI3LrOnEMcdzS1LY47ar3Y1loKQnO0K8A/Fu0OzDrd2kiIn6ATYE7JCqXiQ2ATK1JlE1M0LSCmMIUpth3py5Jj8HGLASBg5BJqgROYTokBBZyyUmkhSV8fpNbLf++KVsMGv4lmCSfplCUn/c3KXDnTNod2B9mWldjaROpo9NzlEq5lHTUG2NbUZanSOL6/jmOrVPrjW1EmWsdlliRT6sLzvMUiM3Ul4jjr2s6oOBZF1dSGnB8hwE5gF4uBZnKID+H68X8t1DgMdFc1B4SC0M1AamABSeEhmtSdY5OiOUQWjO1wUIWTHp3wXY8mcg6OiOvX/Prx40MIM9m63KvF4udfPv1LrMTaG/wUAiVeKDyrTcxOGqdbGg8OGvcOOqwrlUxfIMsEpinaQ8xcPeey9PHIpQeq8NIkrrMM0cK8uszS7xbCxirNVbz73dssFotUZSIua5n6WlbqRkCg0OUMfg6cWW0bQk/wudUgHy8PeLLI3BysfIcgEnA3Skw+E9+IzEM2SIuErRQ/Me/nKLF7L4jUV2jQ+v0p9DEKFtZ8IDFz3zxb4XCyZURorT+w9yoFz05sRHNgaaC1uFVfWz9w8CYcm9RN4/BvowtFWT/Glapqc1jdUrggC1Vrb0wEUFtetKMWOSXGHIL2gjLF8r1ou6ZUayD5Wbby1mAd45qMsHJzVHtGCcVA9qOVep6j9rNB9WdqnDL2g/BsbFuXSZEocMJUPeTMvXYlSoH0aDCmoPiYWqwzpTMRFn8Pn04XWYHgH4i9vjtKY6VdrT3ZpUUdk4nIaV014NBRbVyk3ubo9CefM5PxwgygsqhSJlc+IQ+RWVfzQ3b14woRl7VwIbeQwL2+0GU8KnAr4Wdrj0uFtxHvV2SyIJK2PTTKR63oYay9umuRnmnRCikosqhoSeuv/Swq5VaVYrUS14H4M4KH2dIoFNPZq9ns1auz18FlRbnP2nu4J70+3NH37T00i587b8R7LIUMeSrlshdyi/jl5WOVx8KnSVbiau/dEDsy9rkNvb7yuzX9gNdEjzs0EP4JVfxpRT7EXYMXTPj0jsJs+ucJi4FycTvlSgehhzg5vqdc3Dsm+b7xACOEwZeka2n7rARAxz8guZDNyNeRSwznaCq1XcXZuta5t5lBG2vGi/xnJeWEP0KCuQM+fnZFvqMVKdJQkVD/tvJSUzczYU4rkjusP+vFcgWO6+V1KK6iq3f0DQDXV1ebCYx/0BTXIHpYYgkP3vPDTNahM8OpCJHRBJPqhhATV2G/UrFcnR4GM1aj+8ZDzzZX3NS/Q/FugvfLtfBRgW0bEPsvb/n7B/7+kb/foTmqafpbQXb8XFL6sytU9SOFznrjgg9lNHfZPheFPkZXVKD/n9VMWlqi+S+R0LstLc4iXr49+OhH661VBj386gG93WQbVxhqQkPBXeaKdq/xP8lFm/CceLWZsSDYkYTqdeo/eejyyG6M2+MyhSH/UiLvZQCpF81j3B6XR5+fkdE8YgHf8nFs8IpyPcQT4R9W0zNtz2g/rO07UtQnXcLOyS3CpQvdwWucM81pBCX3MpdKSQ02VIGPxwt4jyK/1VK/Dp10Tr7Xo6chg28G8ESZn3r/fdibCfaXmZxA57OnyJ+HjN64cn5sGsjaQX9LqpqJu2r0KTTO1meqghOdUlF2jm6ekMtSu3z0TVTeLTHT62QTUahoSf1K20zH42o+l+Wd9uk+MQmihi8qo6wQ9L14y9cuFt0NJ7KPXEnA73TsYzsEYkIeHGeDE7NVUReawdCXjAeni911azSfjBxhYiv5PS7mtc0pq+05q+2cFWoEtBf36osJb/M866HhDyc9JOnI9ZGuCZ682nCtMDesFpKj0V9TZ9s3r1MS98VTtjcX8qEmR3EMR7lfupocPePkbjJPfnQNLXSnRuL/e8UY8NmGAPbyzY7ag3O1dn6DchflqkXCCGZrgHXPfukplCPitL85M++L2XbmkTTA1ZVbLlIOc7sR+zU/UIO3NQV3SSC5x29VMHTAiVtPDxu+hCZoaZJDTx1Gm7m/vJA5nJtUsoAjktmPVpEUxcPlPfrJ5F2ldHtPI9PHukQ8pmks+znfc28VUI41aCiqf8Uj+MuubFHCX3vZwG0WvWqodXmYT3xvNZ8q6MVXF3zhZgy1KfK/2dnc5qJIvZNcEAqlQRktSzig5bYRfugI/7xdfvj4nxnpwy9340GGG3OcxwqmE4fGHYkZM8NNDVGEkyxf1IbXN96wLqo+p4Xx6f2Sbi13FKHgW3xcf3YNhqscsSzRjiTux/mjctEc8v/T8zyqCU5Exrx5IaJ48qWwQneTEHswnl+/KcvMo2vAsNZ0kekvvpPQo4SYB+cBl6+HcALq5DhgJuEpo1H2IUbz4Gw+mc4n4y2etx3Pp4a71mgWfZ4IorbmCzUZC5Zx4nJDZfvrc4HrzdfptXfOJrnMBoVnb1GhLjE6z+qjucKpcU65T/ZdYt0Y6nUy79HUrRJP2Pj8d3oLPdrrRjwNj88e5Q+oKo6pCMQxtxJxTNkkjj2nO5daFv8D+T2pioUXAAA=", "src/bscan/audio.py": "H4sIAAAAAAAC/61YTXPjuBG961d0NJUq0qFoyTs7B81qK5vs7O4lyZY9lUNcKhUkQjLLJMEAoC1NNv89rwHwS7JnJlXrgyWC3Y3+eN140HQ6/aHJckWFElleHRLS0oiyLvCdRJWRkYdSVlbYXFXpZPLxWY2WyOweZCkNCS2p1uopz2S2nEymSueHd2ZK/u9W4l3W7CBoHyTxy7wSBVXKyq1SjxTtVCavt6I6FGL/WAtt07w+VduEdrIoaPE2XraeSbJqQuO/xTt6/OVTQrUUj7NK6VIUuZEJvSNDz3mVqWf+tA+0mNMfST1JXYiaogdV07fpWzLxe1jfKc0pMKT29N3K6XJUn6RWs1pkCIyEPd+Zw5FV9t59sSIv8EVYyhRCRXS0z+G+oOcHBce9K5QbynKzExom35MUu4f+zbn9cUTw4CDyKiX6F7wi9ooLVYrHkNkuCsoa7SuU5Vi0xYmecpNvC3kRgXKaJQpQUCSoRk0qm6M6hdhiyTwobXeNRQyy6sxyCPu91LQ90a4QxsRAx9TqvNxoCactSv9jkJ3tVGW1Kgr4/yR0Liq7pEI6wF1bjaSxwyYvZLWTnB02U0I4kpXUhxP8Q+2Rwezcdy0L7PAk2yD68DlvcTKsqosDDqM+lRMOOecac5Ww3/Z0uQHHwgZZQ7hWyStj4TzDhLFhEtco/H7vQN1XGTU7VLAbvANOoPQSgnq/jQJsyNRSAhVDoKDmH0ZIOYcG8n/nO9MQVPTJu6TFzsEA3g6CdiBlGwFBRvklFkFlHsXBR2thqvJplhMPEQNXGW0d/NLJdDqdTPZalbTZ7BvbaLnZUF7WSDhygzbwkAkymbDCQQZWglC35CVKgVYNrw67zC/WWCzybbv+Kx4nk/BQNWV9IoGWq9slo5oqQ10lL5u9t2F2mCqpQVVQpyDYjpVNrYoTJtwPtz9/+Li5u6UVxsp8Pp9MJn/u/XP/KSR66WrpULHE1mmVIe/iFAr7hvaYqvabG7r3Wd/4jczaqWFJHpf4sGM8vOlKzAJubOXVGCVO31hMyY2Ru6Xfp9d3b8jmJZrJa0aho+IzG6jgpkXI0Mybr8VN5DoAAz10Ck8oTkeMrGVy746VjVuJuH5L+KbpN1e8BONSHyRC0D4LK+pyH9Pse7INknXfpzVhqbXPORB3yz0oqlNf6Zlpai4pnODHhMr8yDgtVaWS/viIuND1gzAydp3ZepEyjNn4MSGj4Y7ZpxqbRPDZeY95ktlTLVfTUNgpWr94FiezuclWH3UjY68P3WNaSlFF4pib1cIv53s2+4fVIO5uFhygAqRHiDAyOnahRp1cHHeCbHuE2Og4SCRdX9PBK0PLPcWpMOx0hEQGt701LdGoFQcbxEPJeKxswlj5JKPjENmuLv2j97+GS84yb1GKI3+IrYmOcRxz0Bjr+E6yMJLm6Xy4eXSka6pfcpEVa/qe5l7v+FIU3l935oTDY+wtl/ECWsiWqjfZtoX7it7CqRcDcyArwRYuDyu0B7Oefze57M6UiDNHs2A/puwvFN18S6UJ0gmzDzyBdcQd1tBXCa/ADy4DXIHKFXUYwAK03EILopBPUBQo9xBq6+kWKp5e9CeKgvCMZR0gsJc3lDGUuFQarEtG/P7+76pC2yzX0GSfrgbv8XaZEAv48SV5h7kXKdRhMXfVZ8wf72F7TVdXdIP+8A0Agws5WwTo8Wh5kn57sAotI4lSSweeuM/g/Xw9jNlrxbRa0fyVsNuHey8LA3AQkSwxCqqQiyRsfz9bhLdwjqNfB0CBreDgw+w7R1O2A7dhPCwJpLWA/z+JgmkmxBEFXN6bFla/uVRBxKX07Mw3EiQ5APNV6P2jZh9wToXjqnPLUfMaFN0zC8cfz2miH9egBGys89vt3Wwtz/bhVOeyUeRlPI/88a8Y/HsjLTK/t0zyjMJx8qSYp2vjyzgIGzXNsOtDbiX9LBpjQPJAZOCL5904wwzd/u0O5p6YZqI3mHY1BfMJUUgQ8FKYR7f5OUGiQDUdO9xK+yyZh+IGYV1yPGtkCubPNkSvFc56vzsOwBr+SJNemv2IQIOUcTWBao1uDlkBJ6o1X1xwbTGcdKZj3O7AUm5svkvbUrnPk5v6F2MKNxhVnwanA9DcA6lzirVPgH6vGJ18O8W92iDfub9eMLoGvVAdfFOhZTNVpkCzaAq7wXrE8cVn22E+cAtfoR4D09d0wxMxRnNAMQWbYGhm4UxwXXSKv3SsnEIzhcvixWB210ZPB1bU3hYTaolSz2pwEUvnSXtn61fn6WI+7qsXh/3v1LOXHYs+s/eBB/ak5A5taYmt9JQeWqq9hb5nS3zXakBEMrVD/O44Udq1XbhNh76d9vemKWdpdLvaY109u0sLd6sLEv3AfTB7N8fh89Ndi2kfqjMZ8S7tHW6WQHPmVFwbzJAGpYe3tTH+JTZoHMdDV/3KBIq+icdNwBzHxcBTeuh/D1IgwjC5aoGRjAMbYqAre9KdfwOItdmP+oFtUsc1E+J2SzgR6fyFznFldrxi8CJx6UJPmNQx7zi5nBhtCF4CGmnHwvlhSKdjV1TDqeKI1xe9z6T5yx395VA7g8nno3nZ68/E+Ho0beF8qV+v2LqlOIHdaGbqUS/teY0vbM+BWB4jaYHSzVpz/RAcwCtMjeWIHJ/T1x42r1Mn/ts2ez873Y0mcrzMs/3zCTfQuF8Gk2se/pPxLwfjyp35BWWUBINtztPNxxq8uw5s5CstHe8xwIznMGtmjUg9bBjtbMcXBeZCOhmkWeLiLDEMZORJHlwZEkZmbY6dxvH6hfwP23ZchBEnv6jIuJv7soyJXWurK8nicwX5fHHhpAnogvJO5oVD2XXQieP/p3SwYBEaNwDbBS8Omf+qarrWY2f4APli2ofxeU1HW12x6bugNY7Vi6VgiBLNNub/fLEf/kZ1+QPVcvDzU/fb0+8ExVeg5x0OKcIFC2P5nzho5AetlY7206Z6rNRz9dLvzkv6j//y32k8+R8eoJo3zBYAAA==", "src/bscan/config.py": "H4sIAAAAAAAC/31XW2/bOBZ+968g1IfKgaPEnbYPHnix3Znsy247AdrBYDYINIxE2YQpUiApp95M/vt8h9TFcpwxikY89zsPkyS5+d4IK2uhPSuMruSmtdxLo1fs90+f/8sqqYRjj9JvmdRbUHquC7FgvrValIB5wzjzhwaHknteKO5cNpt96qQFAazmBzp7LjX744E7sWLGQ1p24LX640e2EwfHuAWhsBtIsqJorZN7oQ4sLbZSlTBBu3k2u9kLe2C21ewRtsA06R2rWgVCK5xRe3B3mrX47hnMIwrgWuVhWJIks1llTc3yvGrhhchzJuvGWM+41sYH591s1sEK0xwi/eAdKe0YXCkLvxhRC7grVBkZGu63Sj70xLc4RgSiJfWmh3/Sh0EbhWM2m/1zEDgL/7Ofcf6p2qxmDL/GGm8Ko1bMecvWLLldJuz87w27Xa6YtxT3f/378uuvX7/9yG7f9aDPN1+CxFp4Tirzwu0HqT3wqtUSXpV5D8hAlQQ+3pbS5NYYP3ARQUQWvNiKvJR2grsK4Ejh8FWLAU0FYo1SokzOOGKs3Hx07E+YLuvcikZwj9PIFESiSkrzmDtRrFilDEjW7GN2HXAGpaN4MyKus2XEvGGuUVQmtXSOUhPTtBVDrENxup1sqM7T2yXbcsdQX8JqrnLEcYGwToFdvOdBg9gDEpUg+G2DnlizNAFUlqHikgVL0EyRFXXtCXAs6/Mv/5uAoPOUhNQlixl79Teh/nLz29ckWlfz73lo9ByzIJq5otZGfL8YTaaGPzFUlbHM1WYnGJnpmNHqEKTots4fjd0J6yL3mr07V8yfTSnUUM2aH5XAgyu4fq2YoTvgYZUqtA7cDxbjaCsmQa2Fosioqig6/7ayLIVGLda9YR/fB0TrRE7D6MEgxWv2zbZigHtBPYlYce/1S5LSmsa0/riafjjn7TfqtMFbJ0TZ2/D+XXSB+2KbO/l/0SOWH18NANUkdYJEEsMUhKUlMaSFUIp9eB8dVna0aykuo7OPQm62Pi9FwQ9n0KIxxXbI3PJDnDUoToGB34M/vF5cb5jgFmPYedOE+WY0G+ubKYOQBJGKe8HbvOKFN/Y4fh8m+FPVMVoby8u8UPKojZddf/O6maSJ/Z2ttfyOwDW4aKQj8/Dvp19//jRWcynQKrXU0nlZnEm/2MtiLFzeepOcS/8NQjBk329xD22NKnPbqpFZCJucS3VHzIqtcUKfxHMwlG82VmwQs6PJzXWuzEb65FQo+vsSPhtbUoYcPqI7On8IU7wL9fX19Vlnhl3hb9o3P5rjce62fnINdHfxFcrXRQrSs+ovOZCESzQtRcVB2BXKYd3hu5FFQ2Q1zJJXmXqCyBWuvdXQk69y9QTj8F71mXyVp8PPZ119VNSf5HjqhKrm7PIfYQVYDfPZClqiAjCQZF2g5uyKVclTAFF4n/M8HsLl299IgNIwiZjgV0bn52TU701O+8mon04v9MclJhIh6cSYhzUsRU7ifvMQP05kwFwEgxakrBSioY+Ux4jRHbFbsD3qiT1k2NNql85HzeC8292DuVMUz6CfM1kxNKR2Yc1M94tOMdflMRwM2Ub4dDfvCYRygu1nR36BpncHg6LMabdKaSdbhZCfOGP5I+whmszxSgSWQJ1ZAWaPmzOdzzF72dNzN7YdXTfgyxrTpAmdce3QRTk/NqNz8cSGrMFCgV67CnLgBeQE54PY4Awgnf2BNS61nQPUSX8GN6ZX9CIsOVaWdCOSc1N08PlME0fnj0wMNUmq5sEo+opGdc4DNioaCyqKiQ7jcGRNDEkR2mdiQDoWDJY2mmwLtgvDBqWTpnGfXPSzAYHC/U4tDdjQ2gQMHQDg0LkEpMYFrO/MowoExHXZozoaVD89zweaUMRkBhGvJmsV/N8xPBYIG6zN8nwYlnmYDy7PVy9WMRiHIP5HHG6sNTatklbvtHnU/ZMF7yD21BnznD3tnpPRnLDgkNFPu27bSdEvx51GdvbN9rKRFG6y+fMgzmGb996myMniJPTpxcXTxQVJW7CLi6j3eX7U2SE3Cc0mWrS6sZUchTfEB0SI7zQIE627UPbo/EnDANXVPXoEe9hQQyhISaslOXKHDrg/aWE87b6RgLu34zhcL9/98HbB3obRGd8b6/iSeHvfP1411lh6vVK7pPTovYzPWQSgReVm9GTsplZsKlK+oHcbjbCuIyguDbkcjRycRkYXJAmUTRY26zRZI2rLMbEadQw0zaseBM89pRrcPVOWTCuTSKI+kN6tLpf300h3UukPIuG72yol8mmZE8VdlAIh9y+nIKyfnw7WvwBq2KxSORAAAA==", "src/bscan/data.py": "H4sIAAAAAAAC/7VZbW/jNhL+7l/B0+IA6qooL20XBxcuULQJrkCxu9hNex8MQ6AtylYsiTpRSpzN+b/fM0NKlp1kr1dcjcVGpsiZ4bw88+IgCD7VRd6KUlV5pm1rI5Fp1XaNFiu12uTVWqgqFe1Giw+Pt6ZZbUSqWmV1G08m19ghLJ/PrdC7Glt1KvKqNeIhr1LzYMXykQ+vTJXla5BNhdXrUletanNTCbva6FLHQtw4rlZkphH6XjePE0dCKJLFlHXX4rCpVlrIHz/8GokiXzbGqpAFtK0h2grnC6Pay7ei1KVpHs9KVdf0omnUo51OJgKfJ7qaTtK82Z/7561+xDPfZX8OvnoXr+y9GD6m0qKBMLVu/NV+F6Unr8x9XNWPjtK8SrxuIvF1JG4icbvohZ5MBhpio+wG6iDl1Y2uG7PS1pI9oPsWf3Gc3vXWSnWWVznp1Apoz5Jye8vpnVq1IssLPSly20bCGqHYvmTlSlSkb7HEDXVnoSwygRJpnmW6gaUG47HJYKt3pikVKHkTkiltm6/sRJJ6eomIO31fbVRV6SI8NmTWmJKFu/34w8/vvBeZqniMJ0EQTCb8PkmyjmglicjL2jQtiFbGuY6dTPwaaQrO0H+9s6bqn43tn+yjdTTLrmjzkTr9+7VuE9yz1bvW7atVS1T79x/wdeBYdSXMCWer6n6JfB8L+FenXvpYdWluegKwcJrwStSHgN/m1Nvv+wnh9WO29u+yPiz825vrH25//XidfPrHDx+uP0UwbdvAuH531+bFsBV2WRY6sVpDoP7cb9cfP/38/p2YiaAnfXZ/GQjxRixxKZFnvf1sjCuS7dYaqp68EWf/z88BcCbwXKcc9gG5ytbTXgmR84sp7tKE4ux76DamVzeNKvWUIxDO8pFwxmTsTR2o5nCvUreKgApeXZhqTYYGKFEUe0/LTmEJLtGalSlicj+iTCSgJ7BsNKQDHJBscU+ZFiKRto+1nj0FXdvqRgGckjwNWN49EMo8JA6GZjeqsDpkuuABslnAgiRPRLPnvXec0wwbiM+c/8OLhZjNnOSL2MK4yb0qOm1lwA6VkK8GIeQENiSMXjJtTD27bTrPFHZNs1iXdfvo9EafRuVWi9+I1HXTmEZ6oYSDr780e8Z1OsSY0IspToQeWLB+1C4hqLEJgt/Z9MDxDWkG4Us2ojDQlC9st4TcQtrSbLVoySnC4UStmtZCHfN1bFVZF1pWszKvZKEruQ6j11iK83NxJSDzJfbAMKkpEwIqPbs40B4+dLkkEojBitS0bkxXLx9lUCi4TxAuhhNsGXgE/GalWsnChX/MIo1GlBE7BBeFwID8JxHQh+NUtB2uP4cCIxHH8SJiRMc6YTotLzhE8OD0vYGoHhpju1GX0vHdxF0N/9WScDJOEfRWPgUuD8NzSZ/uSyQCl6j8qvsCPFnhjUHGKFTtX/lv0XPN9p8BbSg4oC6dyn4FBgp8xsLLE6DaR7yd9GKd8mJdrUyqZeiuQ6bLyG5OGYMEwzWzkwNe75t4o3dpDnBrZTifXr5d/Akg59EZN2MT94k/gdDssFI1axtOvVwFPNUYSs9e/2ON9yoe/AHWpdN8dof3Tb5ObIPVQ6KRlLQk0QzFOTFwGkD2odM+Ccldz3D2nO/sBRFmgygUv7NRmpHEwvEwXUtBuxhsZMlGxPlgIroJCeK1JG3s0+PgGcNWkIupkKtSKZ+2U3EfK0vIK6s69rVTyGy2kbhnbyDacd7q0spw/7JjPo1jdeoMcHB5nKawxRJuCA+ECnh1+IY3NTSdeQvzy/HC/tjhcAUf6csuL7DNXTLhuH81630x/h9Ms0XgTKniFv9GWYb0NuM/r0diYdazusEBBgtyEGcQh5+Mb8fJ2EsTHnZR9vK752MVLuLWEBZ5oKEidnYMa4fbePByO1PsY1clLBkqavJZonHuBHA73RVTLAY/vX93HQzJDS9ivQN32HuU4TzIusNxuQVdQm34vYMTqqBwKDHbETR7vYJR/wTPQpKRlwgCG6/qDqVih9AJXYbhQxSM5PIyc0nJKcYF9BhUT6F0jJ/OxAdkDE/gbeH6lzeiq6myXkIIaom4lEFNusQaCpu+9ZJcFKa+brchlf0DesCNtCo5HefrTUvVUZrbrUsRpibZyHpHwoq/CXkZX4izsdDehF1z5BY9Vw4bcoyEa2bvGt0Sm+GDFL62K+WV+AqldLzSeUFLUHVeYvkivoiY8tmJIOQaXsjQR1lFcVeyc16CGhFvSD3y2RUu315cXEC1qBC+vbzytmNrE7KAPSXMjFqcNjaAHCrh0EVKcroseNpyLxdEokRSmQUPXwV9FXjAoi8kwtMPMjOOym4ZhRD7uLyfbxdYk/3VIu8P2x7gyJB7hzHcTt4x5EbiwnsJNm+pgssAluQmv+RVt0P3bFDXhN/RSqlW7z+5fSpDBQs3QIt//n55p4Fh9/rsR8Fd5aHbA7musirTrpZtd+A56p1kQMSCkGISLVdcF6olXTrYtA85ojwoSA7s0aiJjw/bWj1Uvp6kvcQg/mBMIX0khtxhYWGURqCSHKlTc4rR8DNU4kj8tCvOyXLP8m7E0YrY23TV1uaf9Qz5YwQbQ2FBWSRynQBoN+MCo//gonfi+xl8evqi1V2R/RFwkZe+zPZJ5iiK9W7F1fB3EEqvtoewJcZ9zxE8r12fOcTLUpCHw53mdwsKU7oWvj3bye1GAGcKaNvds/fkZn0epr3PxbkTXyEAJycKkjn8+DIUfxVXiD3qZS6ei4nMhAZEiLkfniwOYEWXfGIa+/Mnqvxd8tg7wAkOlSCbie4a+3J8ZFSF6OzsxkNQqgve6BIIZXpX2Q/tpaTLhmj3mrWWB2frEe4o86EUcK0CHvx4jPpAqh1qreC3/bdhndw94T4jWCBiTTU7prcxD7Og0FnbN1c8kgKMUhvK6W8YUmE3P497TLeFTBRTmY9O5KFBMZRwlB0X/qRrql3oLwRzGkVTMWh5KIr6ZeYW/g8Yd9oAcJEwLv9f7j6Q/LFE8zR3w6qdXXm053Q/ulJgtsFxW+VrrQrYw42flZTpKiosplxsfLGyoo7SX9pVVzPxLTzXlbz9ygWXUGm+aodRxAfdnHmyEQ2/zvzwC5GlqnPb+rmXVCe9r8lCTuIsIw0rPPPYJfzbjRvHDoNMnqWpojDoQrHIUKmql8oCDzVcr/Dgj2ZsbnKYN7YdJh3sJ/PB0guOdbhhkQ56+s5PWHhG+Fk3xu3xiQNGY6ncMNH16mncK8blaJ/12QGpyrTStSiDacJTt+XRC5s4DEfS+UamdQ3E037yX6AQWsM+yu3gemB3mtGRLBKX1psR1ALAKvE9uz7onKSJF+AdJAcO++lBu+z3VjxV+8N0HG2S58+6xtuezX4kQQOPYPHdJCOGZ6uuaBOsS3Kkkazpzu2krlnifbzamHyFRgnGp0RHs5ORd0eiogGJRp5eaY8gB2qelrLsdSTWHAwWpwXP229CKjfm1WGiPTmkFEVm38X0IFHY2ZlEHF1h64gRRQZtwt8v7HFZiwweELVgytSHroNbNTd9S8eLw8A94Mjl0RGuEYb7MWIw+T9hBtCOfzSZrAplrfgny/OTWzvMMVkQ1O39rwYAJl3BkPOvo5vodkETRcoyNCZ4cKWgpl9gPDK4S/L40me4TCQJlW9JAi8pMt/J/U4IZH1MGd5GLu9ny+5SsHll/qWm4uabi0sknX4erwEDQBpCC0RTV2juoyzDFA0G3GmS3//GFE4OVoagMYWmbwkHkcPjLaMxyEnTdLxvlNqHMe7A4ySN/oFh7sCHLePGKj3XuS8Kxu3P6dzi66sTOomDKg9pw7IPI9+h9AHZh8TcxQOsNqI7n0ZuFsD/Lw7weKS/Ez4uEF9lQ/H1ZS6o0fTZ319lNvJMhKF3TM6jiMxnfTtF6kGf4ei0F4y9ejv2zwxsK9MOrGnnlDyVWjlESaE+50iAcE+OHYrBX5AT6Gc97jBoZCpywLR5qPBcpYU+wvuBqEOiPqWMXOp3ZJQecka0xppBS0STqyFs81cCcPJslEaG41cx1Rne6aQcGEFn23CeL15wQzTZg6dBHsrEvUfQ11em5l9yKH9L30Q5sRygyVHIQBqAdD75Dze/QnSLHgAA", "src/bscan/efficiency.py": "H4sIAAAAAAAC/61Z23LbyBF951dMsA8GbBIi5ZXKyw1T5fXaSarWscprP6lYU0NgICLCbTEDSYzCj8ov5MtyemZAXERafohKJQFz6enr6e6B53nvyrxqtNBpWYhsJpMkjVJZRDuWS6GaWuay0Ir5V1uhJFv8GISTyUc7E09ZUWomlU5zoWW8nDD2klWiFrnUssYm+SAiHTBRxEzJOhVZ+i8Zsw9Xr8+ZwpGSx2mkmcKo2Zo3mU6rbDcTUdTgBStYWcnaMKdYUtasLCS7ZIrdp0Vc3jNfb8uKVXWZpJmsf2ZxiY1gGKfosswCS7aMZTYri2zHiCTJBqJsI3S0ZSIBq+xe1PmsqZgPoYspmINsuYxTeql+ugimbvXCyLK4NHQTKTTUMJMPuoac4JFBE9IQ7/P47uorSKbFTSaJ1UgqZRmTRTzT5Qz/WC2hfLM7ASkjqBnDn6isY2xWS5aVImazv2BMibwCNTwreUMC06PjR+EZ5JmVe8ri9C6NoZDNjokmTksWN1ajzP/85QP7M4QiqaFfoUgXeivc2YYfkmorNNuKOoaapGW9kuKW/fXqK7bmZb2DkF9/fctIx8HkN6fkuAQv5CGxrEhGolSLtAAv9zK92WoFtZSshkrLHMZJi1STjygsMLwrhvMMFzG87qsSN3JpJKt2sHvBZjnbqEgUYc9rZ7OoLJL0Rtkpjjddl1lGNGXGrRdERcGzJIqwOpZ3aSRZVDV40VvIHSu2mHieN5kkdZkzzpOG1Mo5S/OqrDVcAEJZn5xM2rH6Bo6vZPuelu0TsW8pVUJvs3TTkrnC62F/0eQV7AN9Ve1QBb1gAL9V7HgJrQHdAnIHbkamrRu4ZVYFg3V2yM0fPMWt+PD+7Zevn9/z3//29ur971PmPNqtdrZwazdNmsXc+VZUNoXmXchPreF4e4Cj0Oi0IyCLu7QuC2KXp0UC5qs0uuXWEJBE3En+T1WaMBSbTHIlJRTw+dOnL2xltObDJoh3zoMQZ5TZnfSDEEwQUl2fryeTSSwTxtWuiHxLNrB+05qkrCOo3owkzK4I9a6SbLViXtTEwrPr6ccsDmkwJIJbsA688gOzQGZHCOSVerIfY6Ptjsm4yfOdT/pSDmSm7BmeawndFuzxdumIUwQVvtv8cmjM69v1lC3evGmprhxxg6a3iDkDG2rv+CFvtcb1nYkda6153CkEmE01ZSlZPSD0ISg/xfEDDPctSTu4CuWdyJxy71O9dRLCTyTMG1ne/KBTL4nBSQwo4Ub6lq/e/IGy/xAcBgeu0VlKgc3r9SnSVtQhZT3HFtJZCNBPuAkHWfvBt48/zYJlIxQVAabvH6HMZjg0AAIv5vO53Wj4LqpQKFHXYudrFQz8xDPK9pat0j0jCd5TG7IewT/PaSQBVmgQCGnID5D3POTC0RxG/Dguk9UCCwasezZn9teDLztIbBE9pNPRPASMELqIaJ9cA+l2TBYruM2mR9lkZ1a0oPXiXERq4L/GQ80u9m/2DxQR1oy63nX2dG4bldVuMh60vnxwDAI1U3m0UG0LkG5bDosQoTCWsqIHy00QIs8Aq3pO3obHOJoXJyP5aeh2UKMFJOYg5zjyc4RogfpOrfyHKdR/J+tNqeTqg8hUz++cqzi9EhmHbw+RrDR7b/4h4S0Z+wEp/Q+xZL/89n4+X4wpkG6dEeS55LVOWjtEyY1B9D6YEItU16RKXxOwrymRoZAEzxpCzJ8FFqMArBxmHR9n2azlAoSbygmRapaYBzNNZ4Rzs8bkUaSaqDf2BJFI8ZVRvGF88t0w8GCt0mVsv+omkbtJBpfCYSemoi3q7hWJEQuNxGPep66iJC67uW5sykqYNxNVN+kGhuHU+yFtr3pp1le6BmtBDxQXz4hmC2PrwNfrk75J44rGSdohiCbY7SoOX4WuoLFhO1w3IL18IlM+hEHjawiaa4NvHnwLk8bDX58H18up8VX7d011gFmPjI42p3WAgarIgfyj9AGIz5B/xRZyhsgdHrM4coxRJ6geckCCF4B+Tiin4p5hzp8xzHelzy4/dQBE8MZNQUrigt/o1r8LglCX/qB4AJqQNQzHIXJJrvxg/z1ZTr9+hnUXsK9W5H6U8HozJoJp5pxmFr0ZG9I09Zqmzg9TXWxjMpMFcjGUubhEBp0ccNOggj32cEpLdJhOTfAjCxEl80xp7XAGJg7PGDch3zZs3MW4W2fPG+U6B2KHJfROOdogXDvoMNUznLejFv09YG77Bik7XobHYJWlYlqijmh/S5dOU8qywOIudT7FYlFBhW0fFL6tb0wnfkVvrW0FoicGAro53zt0ax7CB8Nq5b3yKD8kosn06tobN3GeKVcc06RedHGHl7iBMqoM3HonEe+5H+/QHHrrk0xbh+4x6iGveyeXu7YS66lHWCG5dTvnU7aVWbXyjCbZO7TUbRfqU1ZBw1gLNNhufXD6FLKodc1j5yzaavHITvRugtJFTyDKA6bjOiN92+mzpkiTVKIcdANhpO684CRZ60d1WeoThM2Zp/eXzamNiCcMqbOu7z+QqU0uBTXjhkRPOecD9tJb6NQ77tCU1AR63E37/bWWgLsqWPX7VbusD3Dgum1TzRzeDxNhfhunSLC2VV19qRtJzTbKH17emlfXCkHBdE4c0vEcara0WsVDK8bCj16jgZyCkD2NEcRQ0n6KMgN1srmX6dd5VVkSyBGNa/PHU1WWan618Namb00JhQuEkJZKe+tQIcA5ip8GFZXDN7rCcB5oXI0apk7UzuCEr1VXLtHRoYVAv1jlQBOz/uCxU4OltIpaBHshxM0N4WoeXPfPXq9dD1ze97o1OqgQuaSzDGWHKp2NURG19ZedO/hSC0B4TrxHorIPdyLPvC4dfV+V2Wv2RdEYWEZBNe8m25Kzd4PSo9FLr11t1VBtlJbhLzvY5O+f/PFJdFliC+ywu1D16bqySfrVPVW1j05UuAlJSXhpYcy4TXs2JRUHV209cDOKjOAJuHrmBoh2jC+EXN9j+siWP55vqBltEqKMfwnlh7DYkIywwvnLl+fzp0dQT8e7RhAUnnZ5XblJ9wDudoLKNhCkhjkY3/f8ie57AN22HvMvpux1z2DkVhvyKerGFpfj3t8VMd++Lpn2WRkWRzBL2FSIZ+k/Jt7mcbPnj7d7CHY3qK90W1sR87eGZdvQ92qt56+xRldZ8F9YlW5xHVKYcFOjm4tnxVtcTtn5lF08EezanH84ACbnm8UlAU2fi1w8tMeLLCsj+opg2nnjAweaP7RX26x31S4y+hbgD+/Vp5RA4W3DK/gDoZ3tECzAhC6z8Lq4QZhSABWxqGNelHWOlu8nqg8DlPukUr9X3LOXaBC6wrNtXXZPmpZn28LRDdPFfORip0kbFbelYt4PDE7+THr+xu3RGbuYuyukyXFvpOb9mDMea+ptozPo6YMj/QBBdtvV4LBuoqqReXwHvsvFG7W3H5EUeyQxX9iXF+vldM/Yx7fv2vERIrxYU1EvLykYTsybKLeXHC8KUbwIlm/CRbJnH4EvI8BBQC7cOZsFdzdk4CE8T/b//c9hwt6OuXEGlv3qp4tun73sctPB8VMuz9yHonbXJR/cd/WIGy9w604av7/+6Xn0yccScKak5T8me5fu4sTWHb+iyvhAIO6T2dw1iqAcCjv1oGbfZoXHE7li7zm6yG+mkKH6iFJtV7vxR1Dem2KSbqti+dAvWw7fA/xHr/fpAJ45/pBAee//mNO6enrU6hWlpgO89mOiabQtprQftn6mQGGk6/Y73qC46n3V8/bjk08pyJTdIWnCo68H8HLOKWI4N3jPObVpnDvMtz3b5H/ZGfb64R0AAA==", "src/bscan/evaluate.py": "H4sIAAAAAAAC/5VYbXPbNhL+rl+BYz8YvNJM3bT3ohl1psnJNzd3TTNN0nzQeDAQCUq88C0AZVv1+L/fswtQIiU713oSSQD2DbvPLhaIomh5q6ud7o3Qore6bEwu7K4RbSPMrbF74bqq7EVbiLJ3orNt32Ztlc5m77fWuG1b5SIvXVaCqjFz0W+NaDtjdV82G4wGmtKJbNs604if37xeknSihOoyBymGrGYms2KTGkynB1Zld5VJBGReWpO1NifBDj+Mi4VucqE7sMLqXZNtdbPBr74NtrdQYr3odPbWlrXG5K4pe2/oUZ6sjW5ok3dlk7d3omo32G6cCvGRJy4rCKxEbXpbZk5oC3dVrhVO35ocznjdNkWZmyYzomx6Y7EDNxdZtXMYiHXb9g7O7UQLs0CRm87go+nZGCcuZwJ/r2B9pa/1J+NNhgQ4rGepG9vuOiekhrBGC1KGj8qwB0jEr/96/w4bavLSu7PXe/hhY8gDMYv/iVRWZn8QLvDLQJt1s3dwQmUus0o7F9aFfHV9+Wb58V0iikpv4NcY8rvW9uy7wuqMNcFpBz8iyCShLCgemiS1bQH3fHB6Y+ZsRbfvt+C6rMXaZbrhYDP8Li8zcuJG+C/3gtcVRr1tqwpu3uu6EisGaeqMyRdX374UaZrezGYfbdkbR4EnAAG/Ki+tjOcArMlLNtSpB97YY5q5WyGBp6PdcRICPyFKN78lsxDy9L+ubRJwVKVel6DYjylnURTNZoVta6FUset31iglypq9pZum7RnkbjYb5uym09aZYUzSPX+n+21VrgfmtxgeuJpd3e3Jr003THWIPybwr8uDBWnwY6BY3mOrZQ0gvS42CZCtc+UpkFRkgyJQWgDKBX5kpB64Pfz/gRln+kSsdyUysjCat5jpbGsC05AawwY3G2s2iKta7xUBFbxlg/RTgRBjWxqbDEmiDkmisjIRJgOHoXUHzGa9OtSDQV8LMB/Uebt4LhH8NRg5bIphM5DXQL0iT5CCABHlcz6Q7/ryKL0rs08qN7clGUUprzhes1luCqFcuanbMpf3sUe4NdDbiKv0G/FCSPr6GgFLzX0nL+/jOLAN2y5zJ8k/yMcuT8nR11bXJhaXPxAXomut3k8kY/oOWW083yrKfXQgKrpJzWcZ/bR8EwHS0bu3/55H0B7oQrZ7Ou36fWckPB4nLPz0L3q3fDPm7s19r7gMTbmHDRGZ6vW6MpIK3P10P0koqfPRpniPYyK/yzux4BJ5n1Ip2TSSORehJDNJAZK7lI1Z72Wkd3nZKsqcKE6BPFnptakWMuLvCJ4oSut68snRV1geOW5E86Q3zv6OzoSgkWfHyo4uA83If39cGR13tB/yAbHTcUUaGhXq1njRlb+Z3y261vfwHHI61HMIGg9Zmb6P4jhFLpGvKDIyhGEVoSFYKy7z0Q2CckgGLLHRwErfKi5cMo7HKC4CbriKDlVBFqewQd7PcQC1uqfdUpGY0xFLdcHk/JNhRBns4bPHEiyBAT76YwMG3BJXIp60kWWE03YxzVK/VmN6WsokqWRLPUVZiMo0EkDH6f55Z+Q+jsViIb6dH0JStFY0vL8CRakRkkCcwdmVrtc5qu9c1Aozcr8qbyAdnzEslhFK4oQK4zHN6pubLwUeWrJsh9DuJ0LYvWQvwUpKkiR+WPCGyO49a/+y3OJK1Tqz7UTuqZsOdnpnrY5ckD+fiK/aRGxxDqhREMYnhCxwGvswDbhY+C+PjAV9xBOR9aqIHsjpj+D/+/eM15XXc3Mg7MYgdkcB9Sri84q5+BdFvZsQ4MyKbrhHAA0G5xRq1D+wJIx5HUeZGaEDEcLicwEaxycW4ivRtJ/1XCz/+vJqrIykKDrcRvlJcyRHW+qUJcHUxfHRxv/vdgj4I44/t+MZ54MQzZ5X6XgZSSqneeTVxtM6Uoc6EvIFVgQo8UHuPlVG2+a0PbFtRuSK8382KUvs6sm6Fxo/n9ccvsAZNVSZg03UiIYGF7WULjfz835ss+jseRkLhvatzbbBwJyCMm5v/TbR51CpmnY9fJHiKU+F+wBTUeOSUufjpITAFyJqWlsrXuYeN6JCjwOAjiuq2Mz9lW+wwq0q6BDc/olXy+uff1kiWrh44Drn/L3PiH++/SDkXWtxLtK1MTPOGX9zKmgypzvgf8pmdx8M9JcOoCI63gqBg6/FynG5dFQnaVd0avO1QQUexMWJPy3EhJGFwksk8uFxNhRd15GYcAM6oLS3+2n5IcaV6zjZz1teOZgBXHSJj4CPJP4fsW/uM9P14lfE3yytbe1pidvIIsLOP5WdwGXiEb1R2wvghi/hZO1w3RYPh40PU49RiI3vS2HoqEtlA/1FyU94o3TdBQj5NRrT/dHTpHQ0Ep6jbJfriDmyT2BgDDJoAmTWxvVpR71GjapQtRn7fDF0yHem3Gx7p9qm2i+ucRMO6hmOB4/yaARTnMFybCzPslZGp1GUGzL7tIo80o+TOD1GHIQMGXzD7egZAtAFEggoximujbWTo8MnJ/LJrUfmhxBzmoyr5cZjdnqFkMM95HjLkDm4//JdItgdyRHH6DqUTxIQfJOEUMTDj4RidlSIZg/acpj95f440FJbg9upj1oRja64w/0WIWRR4zAdHRcyYNTZD6qHdv4pjhOtJzdw1nyqlqUgbgQ2L+c8l3HaYfn0NihB91SnlwheOO/vvO+feGIaDhW3qzimD9F0PZqLZx+nRqS+C6f3JVwtOVOU6dpsS+zArv99c9ZLRb4kgerhcYrVgrDqvXKOVmpGTxroxKs+2Doczse0n57Qvl2p067t5LRBScSbthmBoqSXnmpaxMbNurQ+jcfRP3kweTL6R0vY96vBFzcBf/XYAu6TySU1ly5uG+hJjF7EcFwc5kLP83TJhSXzq7+5R9EsHurVRXNxM/8+fxQ/fngtaAwVmEm/Kx7FcvkLT6HPHqbQQnuq0EoP89FZf1xE11c10w6N7kDrTON4wZpMV1XojGgWp+gtnEWEL4mwM14b/SiLMiNHDvtVnjvQRqOTZ9JO/o6du9H739z3nyFvcRKxcTAi3aAYXpy0cxdJaHwu0PhcoI2n/QVTDk8lMoQ2Ef4AGT+qRZNuLhAOPR3gKrkxIiT6HWk6w4b3s/RHu9lRL/WWRkNLpNHg5rnSYU1Gw9tiRN35510J6xfv7c48Q354EAM9bjAbt4j+HMykEenvUv92RuOg9aTZGz20SaJKn3l084uHYcwPKgC6UnRXUYpPZKXIFUpF3gfeL7P/AYES2cm+FwAA", "src/bscan/features.py": "H4sIAAAAAAAC/9VXUW/bNhB+1684eC9SKzuO0xWZARUNsmQp0LVB7AIDAkOgJMoRIpMaSS1xuv73faQkW27cIO2wh/nBtk4i777vvjueBoPBvFjxYa74nzUX6ZpyJYUZcpFp8kW9qtYHZZEoqVlIp5efgpHnDVa8HBATGQ3KPE0HpHilZFannMwNJ6mKZSFYSUIankh5S34qM36QMLEsWX5bMWVGRbUWSUgpL0uaHAVTj/CZzc/nUxJxnhs6HE9ehXRXCPeP/AsmRBDSjazo58OJXSmM4hnCZSuuafiG5hTR4fEx5VIRo9ekKS2Lihj2ek23Fw/OBUKnKR1OjoeJBVDJO66cUVc8NUousV3YmGMj4yzxFc+jFbsPycgK19HxGGG8pIyXhnW/Q/ftHFhGpvRqTEYVAFyXTFFelIYrTaCYleUavliK0KVwfJWF4Ez1MsDuC3BfcXZLhyHVQki1YmWheRaEzkfzEczUCjSXcumfIZJDPvwFof16Oh++e0e+VOZGNksDuuW8KsSS4N7Glkqe50VagEO9AaORWJdOeGdZZrkFkzZCm1+u8TAzBYLWps7WLv94MqkBDtxQVuQ5uIR8Vp0upADYwqF0cW8h3jHl4pG5c9AQhIzcOupsPjrO/FZ9iDFnNVzNSib4urvfMvI0F89CP/JOcFvWpqoNCFCIqpTMHE3wX7G1Jn3DKpByfRTSeUjzBfnaEpKGzQZhXwook8Fg4HmOjTjOa4TH45iKVYW8gDvUhiNTt8/ktUiNlKXuHilVHacsBXNea3HFSEyTqDxvdmXV/no8Hnsf4vNzp33UiXfx8RJ/USEw/372fmbtk2NcnJ5d2otXY+/9x9/i5sKy5Hne240vH0LXxQOPjgMPfLfSjLfp8bWaIqMGi2dXYVOqncHF4WxNarZ26zuwJSqqkcgcnU3Bg6P54zK5K8xNW9+Pa+YvaMziTQoBcYCNRl/brmN7TTCy7DthxFZ0GmE0beXggCZWH+4mz5bc3apGQOo8+ONws2Zoy28DB6vQdjKzrngEYEGj6KRZ/sChUd/fPLzZJOiW4KFWT+1K1FZhg1eAz7cr21ZoP5AUxGU9uECvi0XY/bMQdq4mi82yIqeU3hCb9nqFjfS6CGFMF48BA2YKtPCHtl9JoIvOWan5t2JvvSTwku7zkk6Tr73AxdhiGVpIz/OiOIpGYMsnJYpe8b36xJLniLMrxKb/eP2YWtuozdkIO8JzpFXrMnLfna+o+QlGTFugfh+o50DE6CTpbdw0Iv9+2ovle0PDo24z/xpH1iZObvsjHzkH/n3wzTsh6ijjKpoEUJc9h6LxE2E3x2Tv5PTX/zJ0luiu4QNGbvx1x2ibPZz/ccnF0txEaHVBQC9eoKDpp65Zh/S4JbTROqU0cHfjDGlXNbP+TfqbPkjBrVTw82xIKI+ZPR7tom2J2IMtesS9FUefw8hF1FPTXuwbdTWNvqkZXH+vu1k0+2F3beJ25dv56w9RWBmSnaTAHQo42Ag/5ZU2ivmz3YTkyQ8JCcd/zNvOg0kA3fkt0vCS2iNvr5T3lXXHFKYY8ON2dfhxHeXJyE0C1+NFh8JOO/+NtOwpvafKgh19uazT7NsZ2bIc7j3Sw643tt1wk57V/wjYo4NgL6qzP+ZXJ6fzj1c2gM/uTWba7wxhO7tOd5IK66qx9hn54p2fncw/XZ3Fs4uTy7P+jv6RdesKZbtjY3SxbDfsGb+0rPN7QErN14QL+6IzJVNXJb8G6pBGo9FiNxGO66xITfPAdvliM3CdyhUmXN4O9pjHtbGjfgNJN/O75b55aSLNlyuMYqEdf5Udou06+6JGdzccQ3IltS6Skm8mrqcy67cvBmjQDg1aNHUvHTsm+4LZWYJGBlYQfSV8FlPaZvNaLHwnSkgxcMOV2GzwxfsH6k2wQOYOAAA=", "src/bscan/metrics.py": "H4sIAAAAAAAC/7VabY/bNhL+7l9BuEAjJbZrB1eg52YD5HIpEKBNg6RXHGAYAiXRNmuJUinZXm+R/vZ7Zki9rb2b4tAskrU8HA5nhvOuHY/Hb44yO8haF0bkqrY6qSYCIJ0ybFqY7CzqnVXVrshSUalMJbQwEQmQYut2SoOlGs9VrQEXtarqajYavS7MURlCqZYik7HKxELciKosio0IyqLStT4qkWSyqsKJqJLCKqy/DxgjFIUVWbHVtQh2ertTFms5ofDyNNN7Fc5GH5VhOro+t7StAhuZCGp7UNP2HHCrwu+BohK90YnbEBdGio1Old8Etj/2RNmprFS2Wo4EfpLsUNXKRnFR1FVtZRklWgisJyRlpsTrt1CWrEGqknmZqUo83driUFZPRbABApSL0+UeFENih2TCBiNscaomfMb1n6oATeysVSqS3cHsKwExC8NcFzbVZiskNGOKWtTW4clKaJNiF36ZesbU88SoXNpI3cqkbqmXUlsiXOSltLrCjYJ4fSrczUBX4PeJUNYWFucasIw7kLkSulZ55SinKivMNqKr7/H9MOXCWpUxnx9+fi1e/ed1JYJ/qx9BQ6hayGw2EYt/fvfd948oZSMrwtwWVte7nCh/PBjxtfjvYSKezxf/gHGMx+PRaGOLXETR5lAfrIoiofOysNhpoC024Mrj1OeSFOnXX8MaZJypifiofj8ok6jRyC+ZQ16eScGmdDurRAPgV8kVKg/fZ0paM/O+1WDYIonkIYnY4if8NTnYIw4YfSWmf+ePePPmg/imc+FRqjZCKRucl2B+ZlJprTzDLPtfQzF9KeoDDHi1yQpZTwR/rJ0XQKdvfj/AN9gg2Ks4ArBVsAt3AQO+cNrpZAdLEUUCGXHLmTbQCWwTrlQWmQsh3qpgC+GMLo0OIrbgoeBLVsxXcEaU6H2FO6W4M3XD7IW8SW9EpkwArIPRuDbsCcUL8XzZGpJVsAPjRArGRppxOBl8Y8xNaSei5l87CjztHQXElscxtLKYzcWUUJ0fAELwKVFwHAECYYmlExxeBal4cSPm4Wq+xr9QiK8QfyxMuSyAJhhH/PDuA2H98P5DIxXIYNcDYgQ4DMTEMzqbqX4jnrdiQQKC8daveopXIlb1SSkj9HQhgvTlPOSb1Hh+cTN3G05gP11pCLRYg2rQPk8J7KlS2CYW/NIz7KKQR5C1U4Vfcuh1tNNQbJQV2EfcucWJe147HFoiDMhOYQ3q09VGG8ScgPaHQmWVYlLNeYw+ZZA7Z6Aj1ekjhKORI7h8FrX2GpwjJL97nnEJsodMLeHmdPtjONOYHYZpty7yXid7NukC+QE2jrjSuQWs/ddXP77996tf3v78TiDXSkGZFrmHuQb5myeg+6QX7AY+RXS7LM0+zsbT2x7LTCJkpZFMkif97bm81bmuiKEeiQZdSPJSmZwbOZwnkg68Ku55JIHueaVDvuaZxBqZMSvtwpQ5LPVOCleL9cXOvlw9EgnZLXPm3d5RaNdj5KUIgYw+gUfocImJmMJ7W6QNwlkN/+Dl5SDzlNjkaIqXsMsQ0pJ8AZQeDhBjCcz57FuyR3CTK2mCcnV2mrsh56DfIUz2yuqcV+fhkCTEB9WXzPvyIh8SdOLlo3pGwouuBwmH4z1DajjPryj+1BsK4wEpOPwC+ccFNZ//2OlibaQ9Rx70aCKadHa7dEKwp6U68Yq4liL6V/PZdOHule+0OenybuuyC+HVIQ+C0l/i1yI4u0d/ZbW5hjnvMOcN5uZzNDvMz9FsTydESn7ncORrsg10dQxgEnH4QOaQiOlxyDbmAuogEzodocaMoDz+oGwAknVJ2ZHyTQklDyEm7LbFblvcbEPnAB0x0uQepGwS1A8LwcVTUhxMDXuYw5OQAA6oYxH/VRpSjkTX4cpToSs6qkKNK+ID8oQ6oqoGhKyEa09UmkmduYi2WZAk+N1w9ByOSqy7T+ajzx6DjV92jPd4jYt653U+LDhwK89dYpyICBgc2qhuYEXTLtZ18HARMkhgf7R3NzZjOMgEnxG3OvhGpnFmywhDXqCWhjoav2aQEtv1jlBr7+NlVyZ4D+ihNfkAWO6WIT4pxPSR+kG5wfYx0BnNM2cF98QfSN9RI8NBgio6ERsLHLsezcGjquv8gONQOiq9Nq/VSOT2e+y4h0124Q9zJkKQXCa2aNXTxOtVZ0PrkCUKWCTKQLREl8+fHhB7QBw+KjX6gfaoQX/gDOevqY7zKoyuJ1hp/dVtYOUbd31kJygUmwVcJ5t1XQ4MpMQ6uca4JqOjSwc14rHk7fRkHP6nL5A4egMGlzasvmxbyou2pVeFDeJcc3tBLyGUw4QAPxkkklA8fYoquikWVaIePR4+ESG3Vex11Bd8e9lHZbqqVxSY1r1m6haWypGqN1JxvVWgqNGannSKeyfaYdtpoXvWMtYZzHuaarm1Ev0rtav99ql8tH0qr+RDlW6Vz6jo06pSQug5WvFGOFjJwpdz6a3DSzJdklZTvYU33imiy2RWi+UUFf9EzNvd02Z3jdYbdR5zzAUTcFau2qMqLKYqzEqzVYHb2UtfOaVDOhzFzqhXI+Uzac5BOKyQksJsJlTUUoPChlCu8rUzhbANuecebLCd2RTPbkDcLSOkybgKiOCUid/D5xuQJQ1dgj/G4Bxeghw4zro44nQTk2LGO30fTArmJePDd34Rui9/xsRcRDkPm5zI4w3icBvTwO+nKy0Rixf6a/gCPuznYNRrvH7rqr9rM7SAOaOxSbQxy3bwslp1zrVuZhBODW6qtmwnMw5qmGrjf8/n87mDy6zcSa9nZ2zfuoVKqbRBB+4Dc49Jz8k7t30/GPuR0XoBEL1PkTbwckVTzZ4OTkiuyvMuTho+bVWZwcVyHtD1tUAJSumjoizCY7xbwecTs4XrKy38zcA6aXaIthxs6wQ9PXZkWXGiMolihbtvAB11xBhpv+8NKInxdkTTIL+T7xB7ZtsZjxhdmYWekZrGkOeMqQYlm6p0NugTvWiDoONgzvioQppAhOOgVXMYE386VIcCrlI3v9iDcttylccAYdOqHaIwkRuxpTEKC7HtQgbVY0Q7DF1EsdA9H4jltMhnsEN5yOoI8IBMwJ2CHoyP6KJQNIhCMK1ecCmpv78h0jMaqGzBHwXK9ujeYxci+Kpc0CwMXZbB/2Dl5Vtte5IQ/XW3kzTWs68q7Ec+1sRxGPhImiYQHQcC9u6GIA/P0Gg1fGzqdK1u5dcHFeNnxUTstDuxG5L7Q1eL+ZyCKbkmDauQYxgQLBBYW2i4vhK1sqI9c6ebA79A5+pm1/wyg0PXYHge+L4iksMaoAHHF5VJ17Vy0qcBfIAEUeQaMVj8lLwj4nwczYf86dDblKbsDV2j4IpHPBS2ajM99XfDi22Zw0pR3BvQtCz6RRc754thmynRX/4ZhzyffCUQbHYIhP8SJ1uYrduxmA93/Elb2h2MSDt4a9uf0jnPaG/bfrub5KH5jPVBKghyjZ5qTnXHYk7sT6ihCGclva9SXAwb8RKtIVfDzRCnaZfGMuJTozhiNqhnIlqA83fAeZ3gC3jtuIyYLrUan3y5F+U6he/vg9uLm+y+uvv8zSvfbqvC1sGt0+gdoLer39aD1tyv1W7HnbJFFfh+mTzFiYF4jDyhxQthOs+jaKPbbw5lTygc5u9We54d3a30ehgI9lS+LFpQvdJLQm36Mxqe7rkuo8qjiyrA2PO34uCZVXlZnxtmAYVoNKLtKx5Qr73e66CLCc99l6k+5yy/nIpphfYtFf4NETsJT+t2lIs2G2QEVAFXXi7xi6XLV1ZtjfzY8MgJiixnh/c7Bc975OQblF0WRRtgYzb65k1j5Qb67QnnFVNZN2OR1IfhI4w+2QerwfBU+tgW+k33ZqvxvWUfIHPvik2jf28ehAvOm9Cec0inl6vmenD/g3rRSLaVaRPYGRxfgu8uQZ0/9Rc+jRpDZiXMqp0sFfK3c4lb1KE08bvrm1uwn4g8dPGrg5gLSE5BpRmY0VupLnXve1m7vl1ZstrWuZmRlZ2IZd7LtvX5IbR82Ue7u47mUaAuumhgLZm+C6u3urpZ0PuZHP9pNhMQ74hgBAuezygJeic7ckwOWgL0buOW0Ixb5gDsXj95JLBHSGem795ZF8em4jgGIOhOftZCFvOOIHkSkIlvqqqm7skP4Y+S3AB7VgiY/KqJnhFU+VUQzcf8Wje0py0v/oqF+RMvbcwz8H9Y2R3P9yDPNyRq9Tv8FuyEo7+Ni7t7HdkFN6QSl9VMYdHFbQLqHO/CMGxSjNxurdoiTkXxOaK/D+A/Eoh02u9s3DvVe2Nx/mOKh97ZDkYSV5+7LuYnNI5TNImZ+7MCf5gI+O8uUP2h2aAWwL3XRTHSazuaPzxog2nD/DCmNtDHOoAG55EewFNFgcBjYar/J+KkKItXN/0IyQI0QTJ019/f5chBgf0U3CvbXTXcxn8grrBrPRTK6f8yW3jLclJWfE+j/wGidH4fZiMAAA==", "src/bscan/models/__init__.py": "H4sIAAAAAAAC/21SS2/bMAy+61ewOtmAG2zXAB7QFdtpy2XALkEgKBbdCJWpQI9tQdH/Pj2cJk6ri01+5MePD875T6vQgMJRkw7akgdJCsIBYbA06qd75fQfJBjlEKw7rTjnjI3OTrDaS49GE3rQ09G6AD8eN5sz5gdJZ//XX48Pmy4RRgriKJ2cMKDzjAkhjRECetjyEsQ74Jklf/dRGyWmrC+bt9nZV0AxogzRoec7xljqBK4ym2kYn9o1g/T0CNlaUWKAvk8VskhewfwcJh6qcpu9kzQc0PchHg0WntXZ13Zw0EohCaWnvkAXu4PoUXis/vrfvdVYvAwGzEOSRsgQ6JKzcHegnD3aGCo+G+3HXZmBrptKuEG60Q93PXxeL0Q5qT3Cb2kifnPOuqYsAoJ8TgvGf2n95gSWEOZxQ1MGfOHk7e0gM0Ezh/cLAdtPuxr9rurIIz2T/UtQ6OHlrbnXVKDud7n2umK4/wJlU7WrdKbfZ50515eLrozprn3MribNOZ26rfdSIs69DTKpbMuxX/Xz0SWw/9SL0y5EAwAA", "src/bscan/models/baselines.py": "H4sIAAAAAAAC/71VUY/iNhB+z68YpQ9N1CQLgUMcp610S7XSScuq0q7uBSFkkgmxcOzUNiz013ecBBb20nup1EgQEo+/+b5vxoPv+y+swrjWyqpMCdgwg4JLNInnPc2fn2fAQPBtaSFT8qDE3nIlmQCJ9k3pHbxxW0LFjnGBzO41xhWrIVg8LkJgmeUH5uJNBIUSQr1xuQVbYoMMORq+lbA3mAOXwKTlsamVKlxU8MQOGqU94YEBWmAiieCbtKhNjZiVkA6Gn784MHh6nM/jBvFM3itYxcUJVAFfX743mE38XTpIh2EC8M0CN8ByVltKXqJGsKoBM+QGLEcRPEbwuoIMa2M16eWy3hMLAw8v86/PXmAsKcsiEiEs625x803CZe7gtkJtaGeFTP5mLL3CqlYOi/gQy+0X4A0N6/IXijgw6flOSGzsSaB/0ROBVJQcNieLMR7JWCCneVULrMijxmMnlskTZCUTAuWWpJwMpUw83/c9r9CqgvW62LsirdfAHRfClITc1qiLsafa+d+tL1jtHj2ve7ZKZ+XNQyKls0VKz/MywYwBKn4gZbJQ+V5gOPOALuKwuG2SGZhakAHEV0oUBoKcVzAMXStQawHJOKBpvNwh1k1xsNUbv3GDrumSRprDz7GgFtNvTOcBeVZEcJx1/F5RGqVDiH+/edHycheVbwP3cEyyci93QUrV5NX9MLwEaCTSsttNaQO3IyS9LuvanYsg43JGzG1Ex2Rv0+73rrk3qcmQF/xrT+w5E23uDvZmxRk3J8A0d5AdGgFFULM8p0rc7+DuDtIwamwOw4vrrm9+sL0huOaS2/W686UrAdlvNan2RZFlfkQ81+dKNKRpiQ5BrlVNDGZQCMXcu0HyKXx3zuxr1EGYXFK8e+aSJV0u2tf9ul3eqPxEa7cGXELc1Zp7xS2CyTiCTyTfSWXHP+kokVfkR8++UdqGD9vwB2az8lnpijaMnIPvUZ8nEYx6QD9uG097E42nLcSwf8dV1DCd9mfqQW2ou/ge2Mn4AuvC3GfUr/I/+XKF28/2Q8FLZPnPK0pLf7RNFXTN1cI/0ZhjutU8GbTt3b37wHKYB9PBzabpwGm54vQvI8E1IXV3N9OWdACim5Gw+smQOJKsS9MGDdLyusdXYeIGvRti92nYbfoFlg9kIBn5+usKAnZAzWgwK7pDoRuHslN4k6PNnjEbLI/viLEr1TGhf5Hz4+o8pc5ZJuPVx4F1KUlwDBND2fBvDIbh9aDuHOybG//3OP0HtmROQIsIAAA=", "src/bscan/models/bscan.py": "H4sIAAAAAAAC/6VYbY/bNhL+7l8x9X04qpWV2s4FPRcOkE1T4IAmLbJ77YeFIdASZfNWJnUktbZ7uP/e4YvevMruJlnsYiVxOO/zDIfT6fTq+u2bD0BFDtxoyKQwSpYly+GeKk6F0clkcrNnkLOC1qWxFAXf1YoaLgUoVimZ1xnTYJCIqmzPDctMrRjIwn2Tiu+4oCUIadhWyrsJyWTOXmyp2JW0uKuoMgmvzmIbQ8bKEv6xjFZQMQVbRUW2B3OUcP1uppjmeY18tqXM7jSQ5ez1cjF7ved5zkQUTw70NKukLGMoFPtvzUR2BnrPFN2x8F2zEnXj9wwMO1RSITNqDBPOFEvCxe7HRiw7bFme4xc9oWgN2p1RpMW/3LkrK6nWvOD4uj3DL1wwqkgSzxc/RLOP7Jd/z35SspK1mYUlXIjnUTL5g5t9kMH0mkwPrJzGMC2LLJtGMXhz0pwf1q9eYkxgTzXMXy3iH+b/BPQVPTDDlI3K9ZEbywNq7XVwEdiWPjLa1PkZCEWHNpEEa4dRFPXJ0UfyADrDOGb7aDUB/GmUgvaHijOGTczQW+YMut5qZl5o71wX4IJRF2uBaukYWLJLIJiEthBvFT5OYPQHSQMBPHBE+8GRRI4DWppq1uegURf2J5thRGbslHHjrc/2VAjWDy8XmD4MGMXQDlOp5dwkRYq7xMPMAInJBIYfGBCJvlZHrnuZdGC0JY0m0+l0MnE+TtOitj5KU+CW1KBXsRScnjrQmHNlBYT197SyrzFcB09PJmHFSCywwUsiUFMNQkwmE5eRWCrvsSJLRoRI/FMIL6p0/VxvEcWwqt3j/FWMC1Ayqg28DPkJtUC4iBJnpmWO8ICWcvyapgTrrIgbrnqFvjcxtBzdO6yRcVDMxbHGiidR0vKI2iW3ETN2DVjhpOEKL150LGN42dFb6Qm936U2GLgL/fAmp5Wt+zf3u9/w4yIn8wv6IvOU3uWG05IMchaXQh03CsSNYjFsOdXrn2mp2UWm4y4LBYSLqqQZW9+oeoQkMG7ZdRIeY3zNdwfJc9JbibpYFFIdqcpDKE6rkC03TGipIpi9Hnzo4mBBOIYUf9Efp0TvacXaxSN+C94iAy+TU5Tcc3YkdnvUe45hjr/9WGIpCDjBt3DsZexHpq9sKY7l7MdBsa4sEN8vT8vZ1QcPs7332+t3G/gOyPw0B+xL/2EuOSLQe6yXrMYktDseTVou0ou8RQi//ORhaAVbn2Auqs9KZec0q/DcZ9tbfMRk7AkdyothGSPouza0ng/yYchzKwLHK4vnH6Q6INs+pwt6xcrab3iYnw+1XQy0HSr4heouPktdxNl1B2wDQuBF0xbwlVmm/8ptBZtzz/FI1HMyfLMehnVQW15iSJnHUSGU4tNhHDojfp7pvqSbJ2vdk4r2TP9qKOCBV1P0jSQs9pZGOsFtSpEmGUmX6UgejW7QrCFfdOQL64v+hoAYnQjL4btWuahFkZvQiN80Tcwi/WgTlIXBTtLrdl1jL+z5CsvcIZ2nIza689yDjX+2JuEp09LUh9TAEf9OqXkUWPBA59DjM4DCHUOe6kpBI2TvZMRwxxSmUar5n2w996k20oMut84vN352U4lDqNIj47u90S08+pxfDbqI39h4uLUWHeusWM/mEcDf4PbKaXazaTdbpxPXPqIEnU8uiN9uLhOHoKZHhxJD9TxaFG36NG3myh2Cx/LmV8Ha864/Kq+cTMS8n62OtpLse3eC33xWq0HLlv3dzcdXL0c7zsiZ+sEh9ktbVEnPTM0d5rateYBuy0Wj09r/u2BgDwWhI72np3Dm6ifYYkziYijRCunc8bTAxacOewRz6IMULBovsHHk6CS3PWY4ILgEsmy/HmpPDSY6v5FeDCzUjJMtemQLS5aEWYgsunLAvPz7ZhRLQ7lZ0zpfcG2ndGeUN++U2LGmqbK2WNylwViNvK9Lw2dhgtYVHsCUnGWs0saOSO3cJRgO9eoOjnYWxoEVJ6puvnZHtqLW+P/R+mnG1VU7J92imI0FiE8P1b2iGq+fyyp7oqgQrvyYv4KilNSy/j5ZjvAeq/bnVSRGqGSCNOZGsEYZw5MAzvQYrN9pWbN3SklFpu3EZkMZ4oHRtRcjHAeN6eWBzFGkboy38FxXeM5qJY7ROro2A37imSH/G/Ym5LW6hNUBiDys7RFvR7aoHDP02kNlW5H/vzy3Ntcz6rmDXacOthfr8QfCMJPs9c6XzHjhMoiEbPnkEGhviXBgGp3p3I1USH7biDCRwlWBTfx4gC2bR9AmwIBfxKIjt4O43ooNcezxIXh/1PUb36rnnzwgfI2OWPhuO878lW4S2GVB6LJN1/0xmKOhlDt7hXl7tQFSSc3dRR8CJoJl4W8qxnCwyxMfb+9kJ7sHqQ77rIWZrIVJu3s4cpA5K1ddJTibsL69KY0kPKlUiagRloh3aWVd6vYmPWaI9X8Bm5i1d5MVAAA=", "src/bscan/registry.py": "H4sIAAAAAAAC/41WTW/jNhC961cMeJIAVQF6FJBis4mLLhCs02BvhiEw0thhlyIFkkripvnvnSElV3ZS7OrgkDNDzsd7M4wQYvUyoFM9mgAO98oHd4C8tWan9v4Cj0pfHWSvC5CmAzcasvVWj0FZU2XZVRQpn6RP2IEyEB5JYF2Hrs6APtKNOvgLMvUXr0b2+HbRY3Cq9dVf3ho4fr/8Bj7IMHoQDqUW7443d7dX16s/1rc3q/v5qqqqPjg/aNnio9UUhYDcH0x4xKDaElqN0ukD9NJ9x66ILiwp3bPyCD/8Fi565b0ye5GtntAdKGuzh84+G6okyh7yIB80+hKooKPjhcPBukCLXprRt04NgZatsyT687qAVjqn0FMBcXJSgrcgs0Uy8CT1iGRqwCD5hQcECiTI72hgZx3IqWBVJoTIsp2zPTTNbgwUQ9OA6jkGQtNY8kAo+iybZIxFsu9kkK2W3lMwk/IoShaDDI9aPczaO9oe72G+ZNn9ev0NLqMmJ/9Kk/eimniSF9UgHbNr8+s2u19d3TY3X+7JPJ66YPgj6CKuCXiRLZD/ge2SJFSB7NMx9pxi/xvN5Tc3YpFFEdyPJtG0tx3qmgrv4nZwNtjWLiQesauJ4CHtgtxjUsb9JzpAPRMOcdfhDpieuUe9KxJrpnZInCY4CC/xyvoqun5rmrSbPZOAXSYhr944G75ZW9k1c9PmjEUdCw3/wFdrkGrDf6LbTrVhbsPok9GpvNxhw7fk8TR167Ga0wSI1TyfAoIBJNcBX0JeFFM0bNUGpIio+DmFVUevp8GUqWC+hjAOGjcUeQnUulvS5+KzKEFc888N/9yur69uRVFm7/tPmVaPHTZ2YPJKgufBWk2X/C61TylrKsuGYN3Oee9Jzb+U5Wnlpu4fAxlstnHDLeRDCSpg73makfEmUXBbRWFe/Iej2pExUCuxZUowDkqW5OehRg2ZX16CWN99+7L+yknWJ0lS9YMyIx6FHI+K10fnp9Yx2KTbCGYIBVm/KxrlV8lhQNPlVJWcbSPhxLaMB2e+xb0JuS8YrKJYsobumNCeO5jZHenPc03qplNuIiF15tTQJSwm14nFWTdH3BbEWJB5RjFimPwQNdl7NiGQs0AsXxTiKb4QxCdY9XQDKyvmgM8/PHXC7gXIfbXHkIvlYDkDzkl+Pqi+gfpl5Zx1+W75CMUBSoCNht9IrzqMY55Tml840rJlDa/uTRTnsyK9iFTsNJ0ombPanldl+KmqzLcv30uCbYn98aErIyATD9ID1cRH7v+b/mfblc/OSNtnf9qP8X8M837OvL/+8lywyJS7ekjsmzlMF1Vcs0WxyfncLK/c9qKG2YoGU+qbJIprkh3bJ4nnLWm4IycpLz8aZ/SJODdmO16XUUbVFXWMWnDYce3iuC4Y35gKUhFBiLeTXuUcsn8BD3W4Y+EJAAA=", "src/bscan/robustness.py": "H4sIAAAAAAAC/61b63PbRpL/zr9iFqm6ADYIk5KiS7hLV8WOnEvV2XHZ3nw4HgsFAgMSKxJAMANJjFb/+/66Z/Diw05SUcUR5tXT0+/uGTmO80kqPdbZTopSVrquVpHOilwJtypWtdK5VMoTUZ6IuKhzLas0inUdbYXaFJWOay1KTJTq7yLLU1nJPJaiyLf7YDS6uZPVfgBVZEpEZbnNZCJ0IfRGium1uP2f30Ql46JKsnwtXt28+fnDDY/tikRuv1aiuM+xi8RGMbDBpJGr5Honc22gqngjd9LHVtGtyItqF20zxUO+SGWE7aXyfJEFMhCZFirb1dtIS+Ai4k2Ur4FNlpc1jazzaBuM3hVmb8K3krqKslwmgRDfb7eiAi2KHZGFRpWUCZbjkN0RfKEKtFS91diikoJRT+o4W21lMHpTVEIyabpDG/rmScbnobPfZ9jmXuE70pgYxZuOJAw0LnYlfieYqc1YvJVRPiqAGmPQwrNAaMqAiqLGFqCCmxeWDqJIe7swTbzmBOA2NovUSNXAhRC+laUWBciGVare7SI6EoGBMCQyzhSdJd1mpaFCtGWyWEAFKNBSvyWEgth8aOWufwQ3keW22BPTx9vsVnqzkcBPXmRKhiqvHi8m/nTif/Mk+Od+k2kpfoxqnDTKzTRhibDO7mQuPr77IO6DKtABd3a8KIt7WTHwXXkZPl5f+ZcXT7e+iKI4vLy4Rf+2UGoP5HBIom1V1OuNSNNdKdfChQZgQDwnEuDDY0habmW5AWvE8OdbK/wq2kErsDlggbK6ykpAuJxMxpdXk4nAnBUoPi4jpRheDKqGz6+TVwfwnl8L9K0hr0YsNlGV8OSSpUyLtIYIqziCIAphFNQwm3Rn3OiONLuc+8kg1ywcLGeRKKFgYguZ3jZyRHoOoo+LUqyIUrnSVR2zcJPkkIEgqn92lxbtSu5wIAgV8UltSJYgcZ3MGPGm6UZfrYSRTsbbhmKqhKaGj5Pgu2/8aTBp5KRPe57SnMDVclcWjG2ZaUi87dfFWgKPyvCVh8LH51N/PH1SusXdLFGbLNVEAGK8krtM48MXSV0ZuwVtVLK6k8nIYAJ2rMJKA8lLfxJcWxTVPseGOouFfCgBINdZtAX9IF/R3shMsRPZrqy3iuRYYZKSo9Hrz9prCCrblKjCeRRjGNdSpCx/EJ/3mwjQLg3RwUBdFdstUXVgh8EaKDKZQeCOTYz9UAxLHbLeJ620dsxSk7mpxC66leIOBgMG0io23ITMk1BlW3Ip4SS4uhJdv+C2ov2SbJ1pOp+ZCcYB/C8/ffooyG4TY8fNEJCy6lhlu7AZ7omcOU67sIO5ktviXoyvJqRelWRtMaCiJAmTOMzleii96IcK/PAaKKZKso0cT4LJZIrlcD4ZbWM8IHstxvquIItR0aoWVYMStjjQDlWvgGZsLNoOhh8O2zIjyvcDEDChENhwfD1JXr35OEQRnYJ7jcE0dtLdRepWHVLWG43+qSJSDdJi8ot1lZPxrnNMhTrqotpb7pV7DWsnxjuxgrHJgy6WEOMxLbDu8QW+1QueE3ZCFobvp2FIrvXqQvx/ayTGYwVFZdsDuYajDuHDNbTCtl69Gb/9+f+69tubd1izix7CFCdQ4hqWdDxO5F0Glka1LkaO44xGaQX1CcO0pjghDEmToCmgYl6Y+EKNRk1ftYbPhXLZ9r9UkTffhWq+wBqrIk0P2RLCwexVRnqzzVbNRu/RbHfI6125h5cVedl0lbBB6MB/ZWKxDaI6yYoGwKfvP/x48yn8+MGHZ4qSkAdhmk2IZFeAumm2bpbcPMA6ZjT8OkW88kOkI/54S76Avz5R0MNfN3cRdVk4TTzVQAK1SQzt6A7BUha3g6ssR1AQ2t5mDu3RTamzbRJyn29cUdhsYefXOuuml1l8GxoW4oQRNIN4gE8dwXawzIxGX4HNf+HPMDAeJTIVofH2rifGL7F3NbNmZT9rpdUinO2gMllhF4xGnakh9TkYDmCL7WcoH6Rr9Fc+xBRp/cQAb6qqqGaHYByzCNLM6HFoEnIsQaGE+zCDPAUQo6qK9r4ZnhHePlgE/mnJLT5NN9Hs0gg1ASMZJkFU6chipsVcPDqQZsRK22gnnZlw8On4wkG8xK2ryHla8JZLXmTiVasRwSdysxWk5IfGhICo2EIn3SFVFfsCJsgnn4cNSWNcnXjihXCyPLiP7rBfrzN1MDl4BHZPwwEH0SpP70CnwX0Fy+fyHg9+X5nIxO5LOXfev34bTq8db9QzvlbDAxgwd9GKA4493tPhx9tizRERNSTxjHszh0S1ot14bjyLHMsPaq64aVnin42OCAJO6C2xdCPj2/mnqrH1fwl2BJvaUWU7WqKY7phmT53PI4gjnUMQIhiCjaA9EpvE5cMnhtQpLJi+vHAa58fSvbdiPXDbTWhwKNyqmpGDAPweK3VRhslqJhg8hq7gik9KOzzCa8RInJ/pDQfNKdwJe1nEkgigK0g57GydcC/5XXbgpn9sd/JssIrwSlNQikisQNqEJJJcjtECWK0N4uM5YesCn4tvxDNgT4mq6ZhOTIdRwxQRdu4+eOIfc1p8ZAIeTD4EeFOkDq6dPKa5EP0XtJcBlDxgDs4dca7m0vjiHQemsyVWEk7PeuMYnfmCJhj1lbTDxEyBGE0nLj4oAHEfFoC9FM+eiQtkSw+Zmk89AJzK8fTCBksxBz68/T1iEulK8VLIAD6aTGlDvMVk2ZwZPpjPbVZ6Armz+VyMp0sxByQsm86E+AoGIqHQwFiqGa1k/lGUBbkR7q2UJXOlziOEB3tFRQjIkneakk1jMevv+Izp85yourRS2cbnYZVVbqWvJ1bOfFHlaxZOUzMIfpQ5omTYuJNSelIecysfBLYnDXrAQmIwhnhkgxFsG0D0CFASmpyO5jDLkEO44+vgu8l/f4sOjYUE2iMCmliQs2QCgm4DEewgngeTPl3cDZYCnvq10iQBqt6hi3jveV4ATYE6U79VaM8Sy7rSQ6XNoTzWIZELZ9qcpAeHBCrOyn1gijVdHKGhpz7ScI1Q567Y3kGgbXInw7LYkmkoFHyOpn/Gf4FQhpCWQcAwQlgaot8lPAyxSV8eTh2pkVHSfIiiwwUY54xm2okB2wRFTtB12uqF4/W8XV5hQ97E5RXIuLYRDF1vOuyv43WCW5pTGC00TGgU7xAZjD8/KSDGYJxinREcZnQJppPyPxMuoUmNM+w+pA3VUq6vbo+pcxSr4Gy9eAItWnYa4OXFnwJIy44B2vrO7wMYGSd4GlJb7enB2pNK9cXRHbYAdOqDbz4Zz2nPmRcKK410u1e+WFxOJr6gqhCcq41PqDZEpSFglKp5z+0h3ClrPXcAxDkycz1tcPHti/2XGDmQXS5B9eX2Ky7BkPwJZA3jKR0DqK/RNgPU+QJD+J8HDU+okhFDgzsPa1KDr08WcnHG3laIkcbygfJf15Ru69wWEz2ksjJvzT9trQ6PjvO1KPE5OBhibQMDDnv+EFlMnalPl/SMNtuZB6pcl4iFqOA9F+7FxMQCLHpogZIp1I32TcVLcvSSSj7uYM4LmgMpmhwx/Ejgmr3+0AFNyat/QGuBoWNVoaIuxVH6zMktCD55jx76kBYWbws4kGmKDEEFZjkX1+gUquqLfB4qLUs1V/oPnaotvvUPVp07QDf7AGPS8p4HYvQOwgOOCjwEVTaeWx5Z6D14yEEVoiJy8ivl7j1r0L+z1rg39vAZC2wy4cOrG1MBPDaAxyW3Y2NIqlPkMfKTHP/cBU6Irt8kOOSauPXqCij2M4YeUssTxrIf1B/vdybmP2G+20LcMRAYpbEw1bdnDUOPaPi73FhbjjuzSbfWfTD++HcCHhbpTkJnAbjECf6Y+z67PYgK6/FLtK0lVxXc1Knz25xv2/qXdo+E5JNDIdyHn1/98+OndzcfP0LQFzbggQ60kcnFZNCcDpvfUKuJBPzOh/ud9/X77vMww3Taew+aZwwoVfS7FpX2qWVsxPOpMSqmNbatVnsnweVB+9pZmgLK0AV01REmYJsqft+7NmJHQ1dqfO2A5f9Yv0Q0DR5TydJec7Jjo3qOTOjaApkhb9JmhZbZNsQan7I9xu8e2E5DEFgimA6y/O/BqJsDHp1Sb/9Q//yBIvl9gfePhJQyoL+61qZAT7pZIC5wNZNKGPgXJhlSJqrlUBWI6n/eoD6FrCreGN8Tp+sER6fCYEAglNsAoBKQKYQGNMoUxBZaPmjXWnEsxtpBfZQZMCeoC4c+nSVHVgTQ9toWDSSRjua2nuo+e2bGqdNZeucLJhz4zJvqa7uOu2khX0nk86Yo207gbp4g76Lt3JZq22HqxKg5GSyGVufpQnYk5DlnaWNuCuf9mq2LjUxF1wt04VrOGEreYiqzhXfr7wXvo4OSlHEXQc+KmA3NvKnr3stsvdEqpGvC+ZsIMU5vfwbGiEIqs1i78a0lVK/TWfZXEBlcr19S6CNjq9PNqeVDprRy+2EA28k30Jt3hX5DcVZjLh8tmKeZ6EMRu4yvx/7OXKmBlLlirnORZpWimFXzxbFUm2KbUIEo53cEno3SMXSeUQf49rkEgWiAgqrrDHTom5WYyvnMYltq92knm5kbEtpnB0cV43Q9O7w0aBN13z7AYCsp/s21IqBPv9hmIobvTOanL71tSIuKbyMPH4YgXOVHC5aSX6vhVeShCbW3HxyBMXZz84sqSCSzpJJB83DEIBQqGc/bsa7P58cK26jsBm3HSYUmusxN9cBQlhMUcC+172PcaMY0Qf5mPk54lv+le1lICoKTYltDgEzoLjooYiX1vUSmo+8LewAGpigfSEnQu2ckRDezUnktoWxFMfLE3+b8tfKOog7jgBwAHFZkKXRyTwVSyl5NjYU2X5SvEEshcpqqc78h7YpwcspRbLllTsXYppgGbsuwZfrvkkMS6JkgtW/Fur0hamX0pOk9K7d8e6YMd05I9JBXb+n2teEAaV0rw25ZKLrUbl9BUXr6ZwX72NHRRZkKySWRXR9envUMs7H+OA/V3OkXOG9+KzaGfDrOJA9sgG8tBojYaJEBhsAwpkLj4+1MLJZ83Ftibg+hJ4MhcZ6LskTOQUpsLw4bifH7i7uE6hTo2YCXO1sKVSwiLkvD4nYJn0Bxz3KQfjSV7H49u5WF5BwgpZPPwzGR+bcDcEwiLA9M2AVt4cYwEp9en6y3jcWOK7q2/mguzNiRtm/s2Pv2nZS27DDzqEwa8l0y12Z1FN+6d17fRRvK+uKOiMvIBpmWOzi+p9NGwLh7ZNWm6dnsJojLmuIDo77kprpHW4dxm7m+N1q1gM7R9VA7edDdXtpbB2OQDqEYqSkTn1JnOMaIrHMYq7uTOs2CFlZFoc+ovDUW6Jmc3IHivDYfEJQOtu8aHDYMZRJQ7PeGLoNm/YCj95KNHFDbotr6306Vjg9zM4fMg4keuoeCO2wOPyC+5uVfC9e8KywoSu9Ny9TBa8FMN5GGfQ0x71+suz1qt0HxicABqw4j9DY2HzUMIdCJCVLAFrfPo4Er7g80t4GPDpc9I5L3LHGY6k/0yOE+3CEtqQbBIbG1uRvuGD3Yo+u2xyoomk0dlsvwsZ0H46sLDD7ZiIJiFaRSy86klWzTjDh3topfmcz52Av+H2DwJZUqO1tDxTuaGMhdqfdDW0ZPYLK8lr1CZ5o9sPWlK+EspYezql7RgyJXUenA7Jm2T0o7ptPLNdk98PP6CHRvYmgW+X5ueeJlT/MGmDVnAy9N+cddrAMTk7j5fJflXIBYez3VpUtH/lg422hFeUyQ13n2aw27hXnmysXE7PPusuUzP3TK0BdrdgdMwnVV1OVq79oNmvJSb3LVTVbwniHF43COjhEFeomDEBqGryIm9+0pX/yYe+ruUY3LIvVCVIs+gKV3YPkVKVTvRYr72fmsu6H1z2SJRoeHjukQPUt5RCgqO7Y3azCqvg1Cz6OASd4xwS0SwzBgb8MAOtfxEkhTfPLy6+wR6dfRLIpOyewGk0OIJjI5iqEJit8DfIwZcbTxvo9Gx8mElFTi6CgxE0eUsdI0M7X3Vno/k76b4hTZDmimMVXVot8mqC0DMRp/CRbHwYTuQTzcMGNojPvh7pz4ZOLXORPmS1sZ0trbE2y5Koqty/x4KSb04MNMIL1GhM88cGbMsC9B5he8Hcz41P0RW6CjYiy27pXgIKpP1l2lxga1TtYlLjdjCG/YyXTJMlL0xmc/hUxLFWBGsP7NoauVRD40LsQEyfVuN7T0LskLvYZhq4MtGpOzsBI1YOyyZ0AI0PD9m7vuLCEwNeEZgV5Ydg+74WE7oV5ndJUWsDgR2kMLNhooGmYm6cIFsoyicUCe+C9BXZ2D6FTMW54D3GFWSTO8zgL+6PakuHmdtTMRvZo/RGgtGBdG58LtT3o55/NR7OMapG1PdxT7twEG+FBI20Ca5MrrbpwoBF3tQyayC/XAx4FJH4S0akGgLUuIHvhcNlV7skPu8TgC9pyeqLFVsulxTjTs3XtBis4YnqENEA793i0czlNA+Ng06YM6pKxMB33wjDiukTTt7bSmtTyjiI6SucJ2d5m2S2BIou02VGVRpGF/dGkq6DFCjLid3esIV0UepVnCtgggzu+5Y4vMZouv6NqVZAYa3iCiYdNyNJkxO5g5PWfCnOavTEKSsZAezTn27Y3LUtfw8ix96Pl5eGgADQDb/hKIYxSOIDEqC9s7kK9mjzMi9Xm0rXU9SV+S28b6tsrCoD2iPO990hr/SRyO2PZZBKZ/BAFr+dWh4Scts0Nfsvvmb5PI8B9afV7evCKGsgKCSTMaWCSkhzXVGRmqvjIr0ua2wZpEmQErPX0c0dBpg2RStOabLljYkVsUbCrFO0HxEMsrYzkKJA/5XVYVOT0Ydp1XH19//y58/fMPN+EvNx8+/vTzO5DtcNMzxKE8xZaSB+XTpiocIbTn1JYi0+ZPM8CL5il88H21rqkW955alS2zR/DkCYJmO+Y6/OrfoRdZv9YZ8tDei9ATcy396F1YhTDGee50xULyBb3n//yCdfgXAIOutzfvmoD7xEYUP9ED3E0BSivA7iX0dG/IF+V8rWN3H2T8Z6G23IUXp0TW1jAMiOvJ5OxKy//efvR3Cp/ZyWbNvRVcpDw3v8uAT6+oOELHQuYtLVVukyvT88q56N0AQ4dpAlcZzeV4Rxujx+YastXfg+qQSdUJAgYg5vxpNcbsZ/t6GsLtJtw1g5YEZ4NRntUd3G+rOPMWe/vHXAmHQUVJ6LlOkgGXaB/cZwlCIXpRY/lWVpQYAGt+cnNFVTWksXRr6faNC9XEQKKQ9wpDplAYkkKFoc2TjHaN/gNjiGpDhjsAAA==", "src/bscan/ssl_probe.py": "H4sIAAAAAAAC/5VY/W/buBn+3X8Fp/0i7WTFcbJbk8EDimu7FehH0N7dOgQBj5EoW7NEaSSVxA38v+95ScmSnOSwBagrke8X38+HCoLgc2OLWomSlVJoJbO5lo2WRioraIOlddUILWytL1mu6+9SsW8fvs6/sPBssfgYsR9YWa8LY4uUabkGpyGu8Ke6FLfs71e/RMlsdqXlPJNpKbTMWCZNsQaFqpltVaHWDPRC7ZiVxjLTlIWNLmeM/YndFyqr7w1rdJ21KVhvQbSRzIhKst/SWlldl6XMfgMFjK5TUg55QmXMSJkZJgyTd1LvWA0+zao6k6UTLVWKZzqSSOVtXW9P7sXd8k6my/lDaeZ6jsNVcXfgmDVSz701rFBNa5mqdSXKwngfhd+lrlklhYohnTEcy7I7oQuhUhnFbFNkGfxm4FLJ6pxZLZTJIQI2lWKH39MlC7GxPAc1yZk3dY2TsRrWM1vgvPO/sdPF8nyeOfvhS5w1L+gIEKsyobPemh/Yh+ULQWErdposoCMvrCXxiv385fX7TwdX16rcOQVapjVEwpsGDxKMZFZHR9LndZaZvyIekL6pywwUd/BI5m14+/YLa+pCWRbemlSoRGKzxfGj2efWwoOX0GDa0poT3SpzYkzJ4XjNEcZbyfkj/rd1Wpd7zimUj/SzP4H3fQZ0R2bkRGGRYr8YsZZd2sUu7y5dKJqd3cCeecW8HaTI6WDzea+DXZ3ijTSw8yWeRJsVNdd1bVmSJFiopBU4mOheU5HCBjzPgiCYzZAkFeM8b22rYTorqqbWFlmoal9DZjbr1/QaxWRk//5vUyvP3wi7KYvbnvkKrwcu1VbNjnJZNf1SQ+d3+d1knQWJM7sXUNYi424lRi2sK9RzR9ZHoqfMi1JyK25LGfvq4ziuLlLT0XdvPbmRpUwtP8S9o2ptUR5opLordK1IKS9UTiaIO8nptLGrTO6q0m6QXzEFE8pdmGezj5/fvP3A379BOgW/V5vB7MvbX99/ff/5E1Geih/PF/nZUqRn8uzi1cXF+dkrsbw4O80vLpbn6WKZL14txJ/PA8b+yJpCoc+xf7TrNeX3O2ihLlehZkOUlWXpRigk03KxXM4Xr+ani2j24fW/kNGonuVsNstkzmR1i2M4f4XkQXOJSCRvkCTvNNpTzCh9Ll0cY993YnS+uyLF1q2w6eaSUXVA4o8R1bZtm1JeqyahvNZiF0/E3fhk7tyLXpwiO2gll8IaUnaP39s2p+6BPF2x65t49M8Tk9152ZpN2BWHk5mjmVniHdboT0ukszosPUCm05tQvLlLyRDmInjpNgR3FCW2Dv0Rowlb+MDm7CGh/hFmRbU6jdlWyoYef9atjCJ2AhpIyp7dRj87lfO/DDLvC7vpbEFySY1eLjm5OERj8+uiRWEjlJ093O4auQrSNhNBjOykfMtWfi+hPbZCErntaOqFDTU+il74ELPadS7uuzl33dx4G5PJ2rXLlpuDIBekRDSNVFkIB6I2bRiNHBIladNixXs1Go5KcU3kgyVGCuywA48nKY3scFgjisOiT49aM01N06VoUlipXaKZcQbgZBzHHDpG6FrfCdO+pXDqTYMWkmlIZtdWyDMGDbEiDx9mcuALfTUq7nAi78jRdJ7OQ8aTRZN9d7aO4DEY5ASXEzNjFvgRhXWcV2US1gUNTpZrkVIzdhvjhXiiZ/IXYJjI0qlwTxBFY8BItLXMLQ+v2DONFFup+73h9fdUWISXr3XdNo5peN1PHYA6LaVylUbZ6nvIE7ldec+On301Y4AkCFGKJFX4F7rEjKatJqSci7ouV4lCha4/faqVfKYJOT00AMzW4cekLBT+d+VY9qQfOizy5QBFnvI1RSOJt+ep4DreLz5DPgF8Hc/XDhV8TUUp9cA0gluHEfVPjJRfMVI+OkjoaEWDKugndPJar1tK7yt666tMNInIUCXdXhgMICKg/p4LQJpVcHUavEhPlQBa15EwAQau8+WLPD3+GOvo104ANoEDM94vJKm5e1n9gG3GwpzwF3kc3jkmP3GrBNxeZkTD5Fmhx6xj1Ncz6rUhzzeJ8zxJMJ2/j7BCSFsJLUbdQKMOfphM/jX0jZwKxi/Ta1IYLu5EUVI3QkbL0kh0/KbtbOimJgpBS/QGuNDr6p2KE7iIPQYtgLMmXO/L3Fi9j9E77+H/qta71TsB0V4oQcsVywOPqh6dwAOuDXxdEm4mkODVdQ6jiQi2Y0w8FdAB44NLDhKTagsZIZyJMPgBhaH3gBLk9dbPK2+eg7Fj9W6FlE80DcT/q2Rf/atpkXnggMJFOaKms7CHeoAvCBy1hVWP6cZYwgHWLh+AuiD2cX8YbqahSXRthqlEEbuG1TcJCuM/LQUbmWDYHzDhW9UaVN/N0DfzB7oLQaZ3Brn90TT7RDU7ZO3RIhXWMW7KHxLngMlEpb9vMSsygj9jpOiMO1hIfdw0N4lBQ+KEyjGXx6MNnXmIzVC40RGgnM4JwmQA2yGd7NvRCMke4FeX3HkB82g6jvO18/A1bCLYBkmECSDJj4hDZRDzk4wfRDQafS30Prtkj53E6wUOuhGN3Hcl981q2GC1d9F14NIi8KgpLXMCXuMZEE67O2G9p3Ml/GmF+y38Ix44gZ0VrgoLIFNcd8Ne33U31W+8GQ6fmOOsilnoAxhRTsE+wk6VOQLOyD4Cjt7yafgJCRWqlYdFupAT3sfR0KhSl++4MyqHQcJRpChKdLlfq9DxrNxv1EcOHYMSsruw8z4xk/X34IWQ+hN2UR1ueyEUxd6s52iP9KFus8LZOuh8TiHuhdByfEkMO7nB8IkguDkEglR10Ddmz1O6jxBHlIGUOujhjYtg4OoCXRnJ13eXPQv7rzj+U8ujQ+f7yXeWFz5jBQTqHLSFzAm4Hcc6OECAy2njJG6a9t0yPWPp4BSu29IJpnNMNlxQsIMVByvROgxeH/fTFM3d5xDnr6fpWVEYxtf5EJdDJ3CJmojZ0Sh1LElTN2HANQpO3BbgpSZIyG90I6HYdBbddDlV+VndX/FD0MTM5U3Q6U5ovYtVjx9HoGx2JKALpJ+ww5gI+jmB1X5SxATSEVYsubjG/184Zk/g+MgsisDoNeEcSMTVLd/3B0RS5MV6fL7hGMefQShpPddoZ8zqOyctJBmy3ISPW0R9ix/Ut7Thdhu54G+3FHmaFWkQH/JHpGmLO80uiPaeKmZ3RDcJWZ8oe1+7yuL2SWgf3YxzBfzPuetpnBP257xra/4iMPsvZX86XDAWAAA=", "src/bscan/statistics.py": "H4sIAAAAAAAC/6VaW2/byBV+168YcIGGXMja2E0fqtYL7KYJGmCdGLm8lBC4Q3IoM+atM5Rsrev+9n7nzJAiRcnxbo3AJs+cOXPul2E8z7vaFG1+ZpRKhVyvtVrLNq+ruWhkrgFL6rKROjd1ZYSsUtHeKJHo2pizpNbNxohStjq/F/71jTRKnP85WMxmb7ZK70S9aZtNK3R9JxKpda6M+NW0st2YX5dCK1kIXxaFyCtggbhWFqg3lQlwfiETdVMXqdLAa0WhpGlFXakZbxA5tgizq8BQmydD9EDUWpS5MXm1XghxbQVplcEpVhwl2ruaDwJBFunTT1dvZlpBphS7AM4YbGSJX02Rt8uZwM8/1C91tWZaYvCT5lmmtKoSRRtBRKtCtjj044fX4qcvr43we9pnhdqqQhi8KhMwVXUvk1ZcJe9VKfWeeKqS3OSs+FYomdwQxy/A210ltrLIU7bUGSTOtyTgDQiSBphmZ75iY1poMK7r1rRaNkK8ftdJB87Ypm/efByIYOawAwSH1BDVqKplwda63jRmJp788X+W1bqQb+WtYiuYRuFRQ/4rVaWqULtg9sXItbLqbHbtDQxwVorYJLJakHfkBuY0I6J3OodSiCv4qvmBsMwP3y8Ssw1mnufNZpmuSxFF2abdaBVFIi+bWreQrapbVpKZzRzsKzzZ4jeyvSnyuEO+xmuPVW3KZiekEVXTgRpoCgD8a1J34kLBDBsYuqPhtB3lqXEYpUJ0QJwDhN4cUZLPYegCXhWR4edCKT0XZVKRL0TsGY4UQhO6QVw5Wh8/fPgM9PtGJXC1iLx5Lopa4tFhsiHrYqtms6s3nz++e/1JXIrQk5vEmwsPB9EfmSQbLZMdPWfnUSkR3fQMh0V0Rqap6yyCF5i8zbd5y3gwa5JneYJXiFLJLE9VZDcw5UR586GneDGiX3ur2afrX959Ji4evOtzbwlmcriXrmTB0vPmewf4+e3Z1Yd/jUBXb957qyFl7/ri21Q+ffn0+Rjl1ePsO+GyGPJfRWHGwdfmpUICQtg0um7rpC5sDhB+Bp8+kQGD2esPH6+/sHA+SYcTx2wFS+Ht2elwDrlyWFb0CRJpgDDo71ARPmni6ImM2a9P9DJh6uIbTD3OPn/86d37aC+tNeWAirUKn/w4m81SlVHmilws+BWS6lLAQQNx9qNoN02hQrwhDnKkwf+I90jyK5sgKCAoTnGMc2XebdOmVgj3inF8CusFOb/xfd7wg/DceQta8wLED0KjhWR+EIg8s3RVgaJFBwaO0QZZE2xQztgzOnc14BjTTbr4h2zlW02l4vnMdwygjNEWSpZIVqJjPvMGjEQPfPwjZTwIou4R3cYPlr39nSY8V/RgAqJ5qCRwyjoAlW8fQ6nkDiYra727fCuhpk5DKKFxXqnIlnIfrCxFAY5IHSvWDh6WnZAkFBVeyNezh5JPsNP8Dzn3BpXdI4oHACZl7ehR/+A5Nkvqa6iticymRC7doQSvl0Mfg13Y8sTy0IpLd/yaDbcm1kdp1XfuR+3D5TgB0xlutb6j1XDFb5Q3/LJGop/bnIIGZ60b4n54sh/qRRQRiyhitEcTBhFeBQsuwPHODz0mRHHWpSdvNXAGkpmONigSKvVxTOgRzFstNlX+742C//fIGRdpym04yGbnkKmulqNSj0JHNB9KZNsV7ypphysrjyNcskcv+eggthaxt5z0EeSgJSl8kCgy74FFfYyiB2YKD7T9wTx6wREKZiGbBn0GfHK6DM8pudvpxS2hF3o23mp5tK8hnm8HYi5Pdj9bcD4gF/LDarFWrX8bnNwFlraUAChG2CGJPZC6FNvlk40WWSO8XXXiZnDP1t8G45Oo76bsbL1lKZz77Z1maV2R6znYxTv/xbstbADYJB/6DpHXg1EF3pfiim1joi4gsL1Qlc/AgA7hfAHoNIEEjxNfuZ2TbrjRNQu0f+Uo4Q1kDOElt49R5a0gLR24DZ5AK5W0mFZnVbMgCOnOWoPzCKCVrJ6gYtIxDdOm/hb1K62zy3NLynIifhTnJ2l+R60493Y8akBPZ68uKACEP2nboUCOIGTHQtlpTStqAwE4CL5XF4gkEHpeLL26OAglihPsHkfKq4unYqWPE/85vWVw3LeTnGJoeNAwiqzek/yvf/GOxxPYTvLTUTOwHQsNUhEKHLrJo0s3+fqGTZzkhzHVJxk8j/qQUSInxL6l4FGMa38kXT/BL/GkucACzwZLaLPF8RcvX76cs2t0kJdcrqhEdE1GJKnNwNJh60ILNmI7zBiY8THMeISJwZ0zB/mPRMB2xBgQd4D4SOI4FeOhZZNYWLlg5wZo2P6AM/c2aQzAEINK0CjjA8+23AXhOdnyEB4TnPd+Jbklwl2vld/EIc1BaV5H1AnxTEMDubcCkbq6HK+ZTZbl98pc+h70AVwowSXbHah+Db1Cxsixq0VbRzw6+sFCmnbXKB9W6xs+ygnIA64K75Ao/i4uTgprSGGxpc/M4ezhCfP9Qjxa4N0pNg7mSh/DoCVolxNyGN9I8SOCToZef3mAqFjDpKuRAFSSdnZfzPti3hc/f1+ZUHQPp1o/ATuJY8deLlC87Sdo/6td4wxpbgsldXU4T+s6iZByIlbDzAqOiEsJ6OeDTLMjQ+3CfHWobZvDR3T8HSsLyIg1cWQppqXgiEXJpBD5wqZ8S9pD2kfOGvCG5Ph7eCP0PUfhyxWY6mGxg/0RbgpU9Jt8LqKB3ocXEz6rce6M06WmS/vHZqVL+uUusYic+gY9vtx4Hj2EwWLTpLJV/oNXeZz/fJLwa0DdBJmEclMauseVA8Z7YMxAREErSZIRNpQ4wJrUjf2urlIsWV9jOJeJJSvRc8H2mz3lt+5ogjUW1kR0XaSOHgfFsDhsWLI1GdUWUpaog8cObtmg1Z49NeDPLTj+yCrTI1Gc+UjrGIm0nZBVbmL16JbifmlKpYtpGWkchvwe3WnITP1mAv0eQIn1/Q6GYY1x+h1j6OqJMxu3p1fs46ggUyIdVuBocJd9ahg8VoD/8IDIr4quYVFN9YIK0n6qm86NjycGx4SHQLUOvYEAwz6MkLinsz2YvTK6vjjstJBvKXuj10nCF/LF6kgrOHer8dHVwz5R8uRimeskBSvxFDxtzJK6Qu+6UdQGU/tLJupmIcjYdnfUaFXO3HUjqKfK5OtK2Gs4YHezzHTUfMZMy9rmroD7M1LPvOtkRrnpyEARDmZvbqjwcmxg2HeLrixnhD3tFAcdQpoF3H3vswfJkWZ75r8T/6yL0n5n4C5O3OVQR2U/EvR3lvdOC5ks82Jnv0z0/tMTS7OwPylCJS9ZoINxhVQazUWe3lteBpcRvR76drC7qzgxtzXUmWSLok5C0BtmyUETk1IPccn5Z6z9+hbb/wv2cgMG/WYyvNS3C1ntjg2LtaZvSSyb1Gu6HvGbsL5dTc0r068WT5VNu/NB0mxKP5gi0o1Ftzo1/6aq6BMKmvbFy6NDk5YV5lwenRTkVpqqHbN5Ykjakyzlve/e0BPnlX++wKTgl9SxgGogvhcsXAj5jk9MEBKLtpmuDsfIwZ0P1JBtioJrL82h1jWCo+h0IraA9GT5qMn3Dke7h8k7zbrrRrpqj+yNRGSv2n13o7ccRdLc3nAMbyMB4n6RZyscwuPpqXQ+ybtMjsdfS/f35luzoXTreA07pheWLBozfgjEn0S/1IfupUspwerQvUHUeuXplPqsWyD0ZHkV9Rc9wzt9lyyfGOdIwT5xYt+DI/c3yBZc5ggLaUCTek/f4NCIEj3o8IVlCNWHnULTYG5N2F3d/P/jeAnBfXaC/axpJ176okYfDtyHRo+e+WOj13ek5W2aa58+IVetufysN4o+wUEdUX3Lr27WpgidXEC7NcpydAdPh9JnikM8d/+eV6m6767eu8R5pJNxiwdUp4inyFrv5i+D/BU2Ij/SdVGolPy6VEVUVwXf4RQZGsP+JamqiCD9yrosDzaz9qKbdTyEDyKMTTu5ORq4ydHo7yK9C+9gLHvmjXZ1DnRSrZomi8xD39kq8QAaj0vxwFeXwWMXvuxqcwtuAJ7+74i/CRsMCglo2CnBg034wq69WC24UwVrm4o+4RDj1IT6Ad1oz2YI8Cjii4uIcoAXReStUeQt3f0Hue7sf4VURxO2IQAA", "src/bscan/train.py": "H4sIAAAAAAAC/51ZW5PbthV+16/A8CWUo6V3nYwfNpVn0tTJiyfxuOn4YWcHA5GgxC5JsAAoW1H93/sdXHiRuG4bze6KPDg49xuwSZL8rkXVMtVK1qhC1nhi9iCZJXDV7pnp6soyVTLBOq2sylX9AzOylrl1iPlB5k+dqlpLW4+irgphKzzWyphstfqHEXt5v2L4dCd7wMJNw3YmF23meLCbm1y1ZbVn/su8dIscb1arupZFdhJNzR4cdmakLLZ3r75j4CIyA+6N3Cpd7V8blmXZ42r1UVdWGmYVy8t9pvuWF5VO114EzyP7p1Hthsn2WGnVNrK1AdIq3XBjhTUB4JjyWu2z3Bw3bCeNzTob4aZvGqFPDne1+q2tT0u2q9qy7mWbk1BY/SSr/cEaJtrCvRNPWM14qxHzytgqNz/Anp7gamLVK5IT+3uvVCR3LYUmEW60sJKRmYq+JpGIK9ZA2FjVdQBlq18Vg8VspG2YlqJgB6lltkqSZLUqtWoY52Vvey05Z1XTKW1Bq1XWiWVWqwjT+05oI+O7rRo5LLZ9052YMKztIqiDQADgpysCoyzEQ8B4+7mTuiIn/VTuoZkSBfcYG+ZYcXWUWleFNGE/hUbc/bFqC/Xpb4AYCb/t+qoueCmF0yQXMMzU64FAI62GBwY993st9zAk3514WdXYIqWOuJQ1A6qn72AbBFvfWg4ZBQhKbTY+xSL7yK231UhgEpMcXlbQscqfeCGPVQ6+Rhwl96FJmcAlVEdWwY2r1aqQJWvEk+RkI6nTAhx3wuYHbqo/aPOhL0uSHn7gn5R+cjIRnQ3zDEKWRN8pnR8cwAnqXr20MxOTcd85jiuHvGfbgPuLbCUCUCH9/ErWiLYXNSeeKf1Z+y0kOdStbOrF4lURRKFP22UaYaKabNjHvmUjpqehJYzaTqS50H+7YIrtkkm2U/MMQsw/SBzeyEbp0zb1psvsqZNsu2VJ3hciWW/YPmq/RaRGaaEjL9ute3iWOBijBlAILIjE3rDbdfB2p2VR5ZYKVGVNGuLOuz/6dMNE092znVL1mt28IWsi5bQWpwVnO4gjk0lUneA31Vv49OHRvXyq7CH4FwGKIoEy5CI+nbisVJpRmMMDHD8oWEGq+5nODgWkz0/37JhZlUaRW9XyXa3yJ0T29nfdy7Uj+bRhR6Ll9mWo841J119mJCfiiR7tShgbqHLy0Na7h4q/2KG5bGEcVxSXvXh/5SJvasjsrJQ6SdYzLFgrE10n2yL1yFkJ3W26zvKux19XBdP1ehq1cApqWo4a0+I3BYk1q0pneFQXSet/SK1Megu3Oj0AcWS/exWDgVpdbD0pet/9dencbzuNTuHiIHkv7CF5JuGngKxtXc1ufXRc1B3ilI29ecPG94KqXgN5qJ+tQ6KTlWG9SVVLpztcFXKo5JjthNpXHeVtWYQNQ9MP4Kx5oldUYpjCuHhCAHyGXFw9+fDyqsXq6kVSnJIrRSqD8ktwGoeH5HLD+cWLy9JNGxMvbXKPdqtDHBIYygCGv18i8cthJAllLabIvHU4AR3IC2JvqejC1Rk8XnLXeWQ0gNVkDSAsdD9HiKo5hHJ2RmqExKVwwa+ncRT/G41xVHmGkPV7lsVlN9DEW9Z6tcfenHo1BqpuEpwH36VT3L5o4IvZLtqXqAKH9CN+s3nhiiXthvvmSN40F0gD7Zp8N2nKgd8o+LQt+ciM5sxmffoyz6a5QiJcMnIabdjr7zfsZ4Ea8v/SHVvB4HVf8sbQGyu2F6P1ww7Z8XL2SSfBqjo7jAh4rprsx0I0Hz1KNtkDU9d64uMaVvbjM0pHLk6TpSk4hAENvRd8as3DLCx19gHfuXz34bf2fY2iK/pU0WhPUmyTxmeCyKl/j2w6j8n9wnMN/L9+OmQI9c0FynHJK5HjMENZ0GZ//entR7S2d66jvMPZKo1qCigz6ImSkv2iRfF3B04Xup3fhnxMy+RM/FuY+wv7NzsP8REPegR109Z5HiQED5X87L8JEnx/jlFwvyFoMs4EvsCgQ1P2ALGWrU+HtUdF7ZgvuhCmxZisdP7ypzAuO5UfaL5DyB5QyTGKwQi+0yYovzSB3dxtGBpmGFxArSo+A8mRxfBSyM+rOKw4cjRaYNLcy/RumhduzWDkvJvMAyTBV2ruOEc5GhOoVVDBnTXRTTAFUim8zW6dpLfXAxTfbdgpjFCunixMUHz3Z2covlueosiWp68Tm888nXUDCt8j9jChW47uiT1yAfdPTGiLc5hxdQYJkk5GMb6D309zhhikcFCNM6vxVk+JwALd0S3fbtnd1TJdS1RtL2cLPgcz9+XpoqrnT5+ELtL1EmbfugdOFWeOgEz3J6y8rjpnTe4a12JxHEPUIdKWRXY4S3TXrKIsXUEj53wNQUr6+3RyGrEXjFLywrQtYV3AEaXBOUTlJZrS5xTxfhdyeDZELx9gXDubnV9G6qfjkMG12GE4HlYADHy92C40vNPp/Mr93O05wXhXK6fjej3ywbGlRHcsKROuLgBSX0seEtEXlaKafUgeaWAMs/0maIhQPK6nAkqcAYkAiOIxLU/EZf1w+3iJhDoYcE7HSG2G5/qY92xQfGSkZU4FIXGFC3NmqJVJvM0yBsDgJj+zRWAgFYBRWr8Q36663oDri/eIjXdQqjUgVCBc8CKmVd8ZqPJAK48bmsxCynE3C0kSZIBdc3PacCNzUmJ5hkRtHutZ6A7xTAbjrKeBiDbIQvk/u6/7V2hwo6nYOVjqPvu+/MKisdg52MqDk3lVdiZ5+/ZDSgZbO9xoPo9fa3aGJA/f1Pqbx/vsDh009YBBPYLfll/MOhnlRSmLYf4X1weh7Z28eT2vY4tt0qeN926A3l7kPCUETc/pOfHHDZplpTsFwda+Ao0gd765DLGFWHp2SAoHKqDOjlvDkSjctk7Up7PwhapQ7KpQw0oEfzM9Psa5avFE72Ngdi3KhJ1FBfzj7B1go3G/TP0ziKWlePKX3kVGB4WfqW6nIRRpcOa5OaZez9klMxqfm0y2bmi/OmYmI2PYbepivzKxv48C7yHDcSYGjEp1FAJrflCjdAsz23W2eeHCWBYo+KEteHu+5Ec2LM1OiDFbw43rM0w8ki/epm/Sw8Mk1x/d8OJGtKjAGCnX1/HJ7HpFF8P1KM1idP3xK4YTHwuC5rh4b539qPc9HcLf01uc5kSXiaLgIqylSfynBbyl5b/6Cm1sMulcoQ831MBvATbb5AUeD7LutkmhrMWMPeAgPbN9xub/7wj60F6Stsv81Te9BxkR6liZXI+ntJg9c1XuF4fXeBt1cYVEV0tIJs7pjMC5u2/hnIzIebg68hZd/QcrX6w9ShoAAA==", "src/bscan/utils.py": "H4sIAAAAAAAC/61XbXPcNBD+fr9CmMmMXBwnDdCBgzCENCUBmtxwaZmS6Xh0tnxWz5aMJCfxQP87j2T73nJtOgz+JMn7vo92V0EQTDnPhJxHhMtboZWsuLSkVPM5DgmTGTEVK0tycXBFCl7WXJs4CILRKNeqIkmSN7bRPEmIqGqlLTiksswKJc1o1J8VzBSlmA3bd0bJYa3MsKpLZnOlq2GvoVstd6ZdElpR8U57qqTl9xaiB+39ScUkm3PdUdXMFmskE2y7H7atnY/9+YlslwbLpqpbwgyR9Wg0ynhODKKU8FuuW1uAibr9mAhpI5Jxy3UlpDBWpGMyU6okx+RaNzwk+z+QSyX5eETwBV2wyQQylIzIZVNNWh/iSXutdFoQejp5dXD66vnJwcvJNPS/NP+r4cZuaiELriUvu0w40crEffpugsmb6/Ory/OT6fn07Ox58BbGGKu9xaEn7iIbu4O1U1nHu39Y3Xb2u2/IgTN3tDz12xhRb1iZbLF7rrwnSZuMxcIk7JaJks1KTsOV7JUkT7YmLgECH4rcDPyGmFU84I1FAllTWhqcvvrpt5Np8sfV779OJyenZ8np1eWLi5+DiATjrw6/fTb+Jgh3mDNj6YLLzDi7pIw3U9Gl+nGuGZdpUTG9AMcLVppdLI3hyYZ0OD5XWtiiMtSpicgd0zJRsmyPPcK8EH6f8tqSC5+bM62VXkWjZsYMGLYu4l1+ntRMWzN2qPcoBZKXIH2+4d+XR/szYf0FIP7aMI29ZrolXgahPJ7HhJFclNxfNo9bRu4E0HQHyRm/D5dILeB+Xw9iU7CnNPgniN8pIWnFagqgRp3YMEQGU5UBIWFc8PtMzHENaI9gjqIjndG0uBl/8zYiT5+FvZdrZSwRMlfU+5eJdOXgVOUWceQHBdOZW5CcpfDkDpG2XBKJEoKMEH/focwAPAS1Cctaq6xJxUyUwrZLp5yesddx4z1AVN21+3uZhaD2dz4Yu0IWQ6wRDpo1pNDw5vBttEbZ10HQDst4WNBwnVCrlBuj9AblcAi3YfDyvGIp6hbfEJDWTZKqRloIwIVZbjeIfCkEAcpDkvSGJ8kagSvHSWNT0LhljADkbkGDvTf7e9X+Xna9dz7eeznem/6Ji+Zp5pWnCNcUfQ6EWTJriS04OVUoDgSNhKOaLr4js0aUvgjEtctHjntiiG4kUlYow4nDCemNQxryHMsOrE5a2mjtmppRjU75mv/gGlzqQjDUjDlHsfhpenpyiQrx/Cx5ffb79OLqMujtff/BulibxopyVRgdMG4CzapkPvOVWCPAGe3IYiizrsRVvFK6pWFs0TpLckCOnjz58hCgfvx6bxjxEY1wDshDe8xpMD1NJic/nyXTiz/PgpA8Ids/z99MPcUUf7eNWTOIvmZlw71BEbma9osTa7WYNbb7ET5ineuOnsJdr0ploCA08NXQVWVUCa0Mc0uTCgDRLRYlRxH0S+ebqztuU6PsMBOEjwUHWpzmJOlyliQUJ+E6urcd3Rn5bXlLTz6hX3ZhcF1u1Ql9OD7YJXexDtBdMQ6Fxf1+yCHlDpatLtUToHp8rGcTzB585fFKy7xutv3AXUJPuxUpTyTDpT/8j6Kr2mwFy01aFOIZEEc3vQEcQA9UOEHdILXlLn5vaf7EXrrRf3LVd51apIveTVqjQnFUnJSP3eQFSwPWWDUg8yEkEI8VD/lsoF8p7vV1LjzQEo4+fcbaJcmjKVhK+f8i+ohaJ2+jn29ZBSwNTd2wW564dwNVs3d+aon8qNEF+B8/1G9N234SOfZ/qFuHy9MYAwb6QVwtMqFptzHH3XjF7zH1JGqxNl15Fjca8MS9LagzI87QFo0zJvIDjrTHR+4l4OfMY29p0u/CwYeNU6q62Ws8RF0YITGgIZ1URYSi3WK64XjEROHDOLrBR4UfZs1LhScYHnU7eP2/D3KDGVVUa9Y+5FToT6VYDmEPeH0SHnC5t4caHh4Cd/u6rbveQPMALZ78Mr26RPPXgkG4A86Y/I2nGUSG75cAcEU+wbx49PUzup34iKRFIxf+OYaMPyXff0+ODj0cQDTeNXY6MZ1Ndxiuiaq59GKBcj1D18PTL1954nrTrFTpwnUn4EDTklWzjI1JHmvOMur1hxGZBcFWw8NMX2fMcur5N9BebIy1o9GPW49X57Ybk/B0w8Xg3XwZ4enXeud7RfbQlVs3VOFpnncDHFg6Ta3gZeZXXsQNeLvnIDZ+yMFBRA5jBOuLXULIPuSP/gWyF1bmJRAAAA==", "tests/__init__.py": "H4sIAAAAAAAC/wMAAAAAAAAAAAA=", "tests/smoke_test.py": "H4sIAAAAAAAC/7Va+3PbNhL+XX8Fhv3BVCvTkpPmUud0c2nrzHWuTTN1Op05nYZDkaDEE18lKNmq6v/9vl0AfOjhpC9PJjaBxWKx+PaBBRzHucuKtRS1VLUochGIOsl3QtVVUCdxIiOhNgsla+G+WwVKis+HnhDfbUBcBkqJhYyLSooAQ+RDKXOVbMGrCpI8yZfeYPDVSoZrdSOCTZQUIi2CCO0gj0QlVZCVKT5fiXqFQatKSqHkMpN5jbkhiwpXMpPqlYhlUG8wTZRkNEWRq1eDRVCHKxEWacrEICqq+6CKxBWmgWRXYhGEa9NSlHWSJb/ICguTJZEKuZXVTmRFJFMRB1mS7l6Jt8Hbq2/yeBDJWoaa6/WlLAtMZNdEchab5YplrmSQtj2QGxzLIsnrVyKkhfPfQgVbeUVLH7iLpL5MIhAmIUamxTKplQjiGoJVkkiGr8Q2SJOI13RZ5OmOFaNWRYqdkKkWixWYybpKQtUVaMACSXDYaA32RKJBX737UdDqqgwyq0y49X0hqk2uxH1S60VhW2gfsPMZbXNGeh5iK39UwVLeDAR+yl29AvfLjGGjPEUQ8hlCs8vLSG6TEJjY1MWvYbn5NSvVr+EmCubUGSepVGLygj+KTU042KS1umIe84HjOINBXBWZ8P14Q7vu+yLJyqKqsYC80NBQg4Ftq5ZlUClpv/+nitz+rVabOkmbr52yfwIMzQBsXygJKnrWMqhXabKwU77DZzNXvsnKnQiUyMvB4Ifvv38vpkzgQlQsy/eHHlZTpFvpDj1IBe2r2WQ+wMwe8fWSXMmqdscjsi+XOVwJR1WhMxyaVS9UGOSeNhczLeHC55aRtQ8hPhF58XNwI26fj6+7I8Mij5Nlb6huGgnWk18A9xUwqM7zAPoCy+GnJI+K+6/RAicwEotNkka+sUc/DADzkZ5FwZZBkBdV5ivs0hP8DUClnQP48zugPTvOTKvsuDe3r9//+MOtf/ev1+9u70bwQLSZTyiHrb0ZrtfCbSM4kk1e+1AR0A/zeEJ6tnfLIwsAfFq/rKDgSkZJWPvarke8rsY7nOVHGG1kkvk2qYqc9thP8hg7Xibh2tcmNWJX4hPER2yhPjuxenXMf/DD7d2P376/uxFpouoZiTUHWmfzwQDeLdbuyc2x2BvC4lDbNfVEMizcODctthVLcT8NRuLTT9edHvqpx+BLBuWVsop91qOs3GGfqNr1R2m+dZCkGB23vI+IzDq8oER0idy9w5I7N4JkH8F4ALWNwrfz7vXdnUMtMsQnfCLIT8glLiHySFwPR0dzHf44WkJw0388HotXVnCtbuzMaPa52JNUj8Ldn5v4xpvEj2oo9mbxSWzVAGBKcXHx6BxPUknAPjeEvV75EMqyFrf8iwODorabFg1ffns7Hk/+gFrfvP7mW1IrvEZRoSF29vWulC6mGXq+T/S+/3gj9mh4dD5CqexwwahxvB7CMaKMDwbu8CkdkyhWx5jwCTHO6vBtkcvBQRuwPeipOSyMlWSwXdq4fwhIoAEclACsDTre62q5IVt9R18W9EHpBRE8tulzHRsSoUZwDRDupg5FR+csPUdJkNMSp5i7HTh5MRIrmZZTR0fSknIacr3CXQVpLBZFHogY/h1k9K3KooiH52dCCO6I1Q1LJjA7HKIoOFOQOsMFPjOgqHGGVdNNH5uc80rfNnqh2j7BmgOfXxXFOTmZsR1fLRXtT+npYEffZlco1zDhmlo9fOsOmCD+9uQDHCWIWz+l0wevympkpm5DT8TZOkoq14T46ftqI4fGgXLqM+16bT2d/ltThUWWwVbhjmOW3rOq8KGK6Z7pbRNMShiqVhOGpm04NjwzhiO0D2Gne1LAlbjglosO2yx44PRF+cCSDuOGP7cSpSZE+uPfF9Ua0XE6PprQ4Ujncbasptc0Srdwou4rZN/T5zyrbtYKme71b4+QroWCnD2BKTu9OF6gQxmDl/sL0sfnYweRzQarMF768ClteDOpgQmGaJiLX9kVYA/oV2fTjRPoJE4N0PQnGwScIHshbxdkqXOUWrlmhz8TLk8scN6YzYeU5NEcn4iJd3AiuvyTfpj/P3V0d/pTfNY5cgmXFk/7yw4kLKpyo6yXIBXWOuHsmkMIZVnNOpy6kILqCicwGXU8LqUtIN0/Ni103FIlOoSrN5/jyQOiYh6k/pdvLr/7/j+9pu9u3zoHaUYMlm2e6YacpCILKodekhbhbDzvkT+gq7JD9FJa02/tZkh7OdOK8ilFd+b90IEzrqTMv/QSFSORq6X7MPSCNEVYoPPUgxcRcsV0SjQxpqufXXNPKnOQin+I8UHEpgCisxPMHG0qTnp9ylrm4lOcjMbj8UkZgoVyDdNLYjMUf5+K65EwjSNu6w2knZipkhK/vZNjHiSPimK4qijEExZg9ZMXFPc1k8dDOyAW3NYAwkL42usf1i//EgifqgcAvFWyfEFHfJyAM7+SJQ4G+Grh2IcymPweIJ+BnIHwMe6AOf8PQo4ON8emE6607ehla8/arJs+u+L3zQZLJ45GjS6ZRbjSh4fp5CTMeAQBmEBOqDDCDwnjXxA6ubdrEZbC2AXLTBITq9NQBt3YGxOA6WSMAyWd22iL0TJBx1MslnT8+k1Loukgb1BVwc6XP2+C1DXRFAdauzqaEru7oFl/SUqXZh7p2YYwLicv8su2clInoTPo55dqBjnI1lhpPbltoimVsSVGpLWkZ55oTraXf+ZPz5KOa2i8j3oLc6lgVplMxbOHyfXLh8nLlzjWx2F4leE/ND4fU1vfrIhj1652gp1gBbZF5plUza/ypTseejhQ5FFQRT4VCLABjCQgRnEW3/rOIXzg2GsPLG/A1Jzr3R2cHXJJCrgOCUe/ST6b/VmDWY/ElrbxjYelZb2croOKradWgXbd/ULCbD2nyE06wMa7xE2TfjA0bI0JjMT6cO/3a519uJbXaUkfDUC0bi1CnnuC0zaI1RQ8xZ+GEDrrSe12TiLGTq13LlF6elfXQahEOUTnQTm2BxRm8PEeeCRmDldlvAWwhKFqOsuoRkN7PhK04/Ou06wp0J8oTB167LPoGQmb1l63XBUdF9pqlktZ5AfRF5Gz7ZXMnhpHZcB27CdcfWXJ4XzJV8FCoX7KKi8BqyLENkW6WE4isRNSuki7UVKYYrEU9zw/Boc1XB0a76ukho0PTrhFNXNy3wxA/gFbgHtzP4dtjtiPRWrYdei1R2U6JLjkEl3k0ryweY9JO7BVC1V5OqUy9I7E85Ggg9NIjPlf94DEhoytBNVOB9UcPsBNqI4SpR3GbEocMp80L+bVN7FuxGMmZimkghlEezYScIT47+XLOTtKS8Vb2CV7PtZUJ11Qb+aTbgjnlxRSNz7EuJCeD2EL9TYl4Cyhuympj1E57QLIehrHbsWN3Qkgju2TUj9eBkExWMiUaHZeXbDqGudj7NV6n8+9E5clN0/et/Aty28PUZrjFfG7styumhsb6MWIsA2qJMDJW7jmxkYsNyA9zPt0uberd1vzL6pwNehbLesYCTkQ/NfjlokMbLfQv2u4fAxqKczuOmOaDl3hOEge6Ziqs8cTfpYg75PfsL6p+ch6H6DKjOeK4FIQw5HBnKv1NSRQaJ03Y8rUHPzSMM99tqIDg+hGBRJ7eJDJ9urd7sEhKWtiAO86fD//Hp5SFCurJGUxEDy+HfReR0H2k5t57R0ABfK0mk7k5bP+YHNxB2y4vEmHvUo1vPPcizc5J7cBAlqSB9XODyuQ+HQ5V5Q7n27ezJ2Ba68OdscsPWsRB7X1ZRVEyi/WVPeC5yg9ahCJ0gUOQIA9jJam8TKayvoaRgof0PvLP1KZR8bonsyDtORtTuU+H+kj8qnpNXEzPZE1y6CPI3J1oGNgfUYY0Wdblpl82eEtjpuR76PhzZUAq5KMinb3+XE0IM7HtoIQZyZh1z95cf1y8gVYf3n31eu3oplQCyDCVZAvMerQNYO3cbHWNVkf+8IT9X3xwbvmzn0u13PILYmP97Cxc3Sd/Vn3ptowRHbXL8t1nSqP/LBP/dhDNsLFtHdR5oZ0obicpkG2iAKhbnSFbtCxPYygOzCPhFWuyxFH53i+2mQZLMyjfoeuY3G4rMkHd7AcrhvrJA5m/EIiuynpOJ0Fpc/ZFmx2am/e7mWyXNWKfeL0TZCqjki/xfdkni4pULTx6VrODdcm3e00dtPbbXCUUz6Z7LbvB44yXs7a+i1PXtgca7lNiE+puLNLEyqC9+5D3WzUi6VbCIsEi3V5EEztH6a3w/b6L2HbHuW6NYJ0AiRekwNJ8lhWMg8lDgRkJVR3CRapdAZPHAbTifFv/TRu5hDao6QiTwL4H6dvuoTug4wyuGI567bMRxqqPrdZgk4LETT+kDut4zreamM0nXyR6PuNc5sVGsO3Hutv3pmHJjfnX7CMmkcrdFWJdI4ra8dVj15W2M7S44y9TrY4KbXPY4IS3Kil6PB+1Uypt6Xny4j176kM6jJd/9nCh/yW4rovnSC0YJ0004Cna7l82io5FgJ9pixOD2x6Pb2Cue44Sv6gc91z+hSSxMIJNiGPzvS7IjrN0TUi3SI687aNrxKd+fFFvpHfVPSyGTOc21oejW+7pKyarqeshzXV7K229X6ltCmr7Nf4z9XRfUsxnRaVqCSnmlMoqY2LS0N9x741WbZNszOrmJOeEKzWa51Ac/ZKS6OrCiyDvsJwUwUhZ8lUw/TJNxlFDR9tNn+wAdagNPqsPb30eg+zYCD0VutPLwMePAC7Ee1ztCYnoBQJCCfp217Kv1XPesDmRB5QUp1P0UOEMhp0o7bS70+6+IQnZdVO6D3G8WHgA5dwRxZqbuOefIJw9qZu1r+XDMvNqdvI+AKr3kPux4tHwLGPmd+cy1jN2GcYZaTDaai2vawGNPqWfDZrWhRfBABC+u/5HNmGzy/UurnOmciGOWfjOQtHL9SGtFJ+BRglMUIdbIp6Hp3j4NROr+PNhJ6EkHVZpjMKufMG5AySgXlfwQChhN28h+DLWbeJzk7n9RP6Dt9CUcLuHNxUg6y9qj66ImbMU1wzz13IYlMcRUkrEQ1FeK5m9rELZ/b6GZFGJ2HTjDTrad5fuXoxI6FhoR9I+LrRZEaD7tuV/+Z3333/71vx/vbuvdhf0CS3X1+Qd9FjZhetYBdz8xiI3ruA6lF0rxpix90jy4XcF1ruC5abWV6ckPtqT0Wg5lOn+4oftdgb9iPhm1dIZtvHHUG7GjSCTgaDAQjsKxxWo+/T4xnfd7RZAzUgvNvh/JjdPiS1q5/WDAf/B3vil86BLAAA"}
MANIFEST = {"configs/base.yaml": "bbbe96f72121885b154898c55ab7ae6ca35e9938887b85664ca0793dabcbf526", "configs/bscan_controlled.yaml": "9a87528a1248046396ced6658e16b8007e999801ca81a8333e70903ac7d1977c", "configs/bscan_orig6s.yaml": "68fa46d39c6b2b92a6994861684ce32578c394579fb35c17b843b73bab2e8bca", "configs/dual_no_se.yaml": "89905fca9ac7e4e27746effef662ea4e04448367e61ba6e2ec8c86d323b0eb15", "configs/dual_no_tattn.yaml": "9d4c88ce969fad322a0c7cd85a3986df68a55b0938539ff5a49034e9a79c5045", "configs/dual_plain.yaml": "a76edd334ce7d894d3861ae275229c4e6560981003850c9fbbb853b5e6bc02c5", "configs/experiments.yaml": "8461649f1b859bfe5126dfa1c7e58b31f586d7304f493e200f8db9558b802d28", "configs/lcnn_lfcc.yaml": "325a93b135ae9529d41897557205febc931fa1f06071021b13486182e1157cee", "configs/lfcc_only.yaml": "5ffe35b8a0e0e29c5b83746cff369c9e7707707a237df293a01bf985993799f3", "configs/mel_mfcc.yaml": "5588f80a7b8fd1287df470b063263a9e56713f38337909bb07110652e94d0219", "configs/mel_only.yaml": "649db7362400be37e4c2c519058b4a4200ae09fed1f740cc0ae203bf4011f2e9", "configs/mfcc_only.yaml": "726242c52148738ea6e6ce35022a8f5f3519e05dc023151d6098b749451df3cb", "src/bscan/__init__.py": "4266e46286c3d8e3966931ce3bf9721b2b37ed71a9183ce6293075506035caa2", "src/bscan/analysis.py": "f1b89dd89520a28897e665704b7a3c80a42e5f45c0987f5b26d6a31af83ddb40", "src/bscan/audio.py": "ea76cbdc5b38652d051c3f952dd603645f62b0b4cb00f11d21aa98fcfe10a1e7", "src/bscan/config.py": "b03ec64b973583b39e3c0f747a5891ecc7b406134707e3669d02b5f29b54050f", "src/bscan/data.py": "b7175c245df208f3e11cbfe81d0c904f8a31928931ed0c196ae0d19c6cc77b3d", "src/bscan/efficiency.py": "09fdaffc5184187d1a718ad0e275dc1f9267175b3aca711298402aa69855aa65", "src/bscan/evaluate.py": "277194b8df65ec6f55e081f5cdffe2bcc348424e3d8bf5421aef639bfc01d3f4", "src/bscan/features.py": "d0c49cc558fe66cb449f942fab4c1db6f3081ae34d81558ce79aaf137fe1dcab", "src/bscan/metrics.py": "9322ee4c034b94c0a0c224def64232fa7bc1a5e6195fbb6ba947e6467dc8ebb8", "src/bscan/models/__init__.py": "bc6172d08fc611959aa6d252c7a53a1591e6b45c612020a3ee02a9670363c6a2", "src/bscan/models/baselines.py": "0044cbfdafbac5df2dbcba291db9ce391c2b043aee98a89574308d7710776b6b", "src/bscan/models/bscan.py": "fdd2eeb91d6e65c762141680931bdd49a49ab2c408f4d22eb978cebea3ca5696", "src/bscan/registry.py": "442d1b8a2075733aa122c87abea6f9c9706fce40ea54b14c28c918a30005a7d9", "src/bscan/robustness.py": "1f83a74c279e13c13c531e0e13d000ef5ff0d222ef1c6246f35a5c20083f650c", "src/bscan/ssl_probe.py": "a976608fccd19b6e80f1e61e5e9bc2cac1c01b193fc4c4a4b6d10d529e482d8e", "src/bscan/statistics.py": "a6c3f8414d56741d9b41b3a518dea7a1c63ee7648223c72b2d08238c447d6746", "src/bscan/train.py": "fa31ddf1b89fb12569f7b8b952414357c9ec390a33d2b1a91a9038d2b7ace1f3", "src/bscan/utils.py": "0b4d2bc08cf2b81560bbdc7f44ac23e5953803d78b13a0466605544db0bd0d7d", "tests/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "tests/smoke_test.py": "e0d4f54520ba4592b53ebb58fb4e6e1847d458110026d25b1ef4a6f3b72bedaf"}
for rel, blob in FILES.items():
    data = gzip.decompress(base64.b64decode(blob))
    assert hashlib.sha256(data).hexdigest() == MANIFEST[rel], rel
    p = os.path.join(WORK, rel); os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, "wb").write(data)
open(f"{WORK}/CODE_VERSION.txt", "w").write(CODE_VERSION)
META = gzip.decompress(base64.b64decode("..."))
assert hashlib.sha256(META).hexdigest() == "a19868ec5ab2a1c62f94b04d964117b7af14756c2ce7b24734665c3ae8e6fc9e"
os.makedirs(f"{WORK}/metadata", exist_ok=True)
open(f"{WORK}/metadata/unified_metadata.csv", "wb").write(META)
import pandas as pd
meta = pd.read_csv(f"{WORK}/metadata/unified_metadata.csv", dtype={"utterance_id": str}, low_memory=False)
print(f"code {CODE_VERSION}: {len(FILES)} files verified; metadata: {len(meta)} recordings")

In [ ]:
ARCHIVES = {
 "banglafake": {
  "url": "https://huggingface.co/datasets/sifat1221/banglaFake/resolve/main/final_data.zip",
  "file": "data/banglafake/final_data.zip",
  "extract_to": "data/banglafake/extracted",
  "sha256": "3034353ef96a9f8c8a8118e65682b31b36de14c4f9159a2ee0e5cd5a9dfaa8b0",
  "bytes": 5642775456
 },
 "mendeley": {
  "url": "https://data.mendeley.com/public-api/zip/4ftmwt86vr/download/4",
  "file": "data/mendeley/4ftmwt86vr-4.zip",
  "extract_to": "data/mendeley/extracted",
  "sha256": "aefdfbf6351849c186717767cd1c901a91e7033ccc1bc6af65161995bb86f4d3",
  "bytes": 519835011
 }
}
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
tick("download")
for name, a in ARCHIVES.items():
    dst = os.path.join(WORK, a["file"]); os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not (os.path.exists(dst) and os.path.getsize(dst) == a["bytes"]):
        subprocess.run(["curl", "-L", "--fail", "--retry", "10", "--retry-delay", "10", "-C", "-", "-s", "-S", "-o", dst, a["url"]], check=True)
    got = sha256_file(dst)
    if got != a["sha256"]:  # not fatal: every audio file is verified individually below
        print(f"WARNING {name}: archive SHA-256 {got[:16]}... differs from the recorded {a['sha256'][:16]}...")
    out = os.path.join(WORK, a["extract_to"])
    if not os.path.exists(out + "/.done"):
        os.makedirs(out, exist_ok=True); subprocess.run(["unzip", "-q", "-o", dst, "-d", out], check=True)
        open(out + "/.done", "w").write("ok")
tock("download")
from concurrent.futures import ThreadPoolExecutor
root = f"{WORK}/data"
def check(row):
    p = os.path.join(root, row[0])
    return row[0] if (not os.path.exists(p) or sha256_file(p) != row[1]) else None
with ThreadPoolExecutor(8) as ex:
    bad = [r for r in ex.map(check, meta[["audio_path", "sha256"]].itertuples(index=False)) if r]
print("audio files verified:", len(meta) - len(bad), "/", len(meta)); assert not bad, bad[:10]

In [ ]:
# Stage A: smoke test on CUDA (must pass)
tick("smoke")
os.chdir(WORK)
env = dict(os.environ, PYTHONPATH=f"{WORK}/src", BSCAN_CODE_VERSION=CODE_VERSION)  # recorded in every environment.json
r = subprocess.run([sys.executable, "-m", "tests.smoke_test", "--device", "cuda", "--out", f"{WORK}/results/smoke_cuda",
                    "--metadata", f"{WORK}/metadata/unified_metadata.csv", "--audio_root", f"{WORK}/data"],
                   env=env, capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith("[") or "SMOKE" in l))
tock("smoke")
assert r.returncode == 0, r.stdout[-3000:] + r.stderr[-3000:]

In [ ]:
# Stages B / C / D: train + evaluate every registered neural run of the selected stages (resumable)
sys.path.insert(0, f"{WORK}/src")
from bscan.registry import load_registry, expected_runs
REG = load_registry()
common = [f"data.audio_root={WORK}/data", f"data.cache_dir={WORK}/data/cache",
          f"data.metadata_csv={WORK}/metadata/unified_metadata.csv", f"out_dir={WORK}/results/runs", "data.num_workers=2"]
os.makedirs(f"{WORK}/results", exist_ok=True)
todo = [r for r in expected_runs(REG, stages=tuple(s for s in STAGES if s in "BCD"))
        if REG["models"][r.model]["kind"] == "neural"]
print(f"{len(todo)} runs in stages {STAGES}")
def on_drive(name): return USE_DRIVE and os.path.exists(f"{DRIVE_DIR}/runs/{name}/metrics.json")
for run in todo:
    if on_drive(run.name):
        print("skip (on Drive):", run.name); continue
    ov = common + [f"train.seed={run.seed}", f"data.protocol={run.protocol}"]
    for step in ("train", "evaluate"):
        tick(f"{run.name}:{step}")
        log = f"{WORK}/results/{run.name}.{step}.log"
        with open(log, "w") as fh:
            p = subprocess.run([sys.executable, "-m", f"bscan.{step}", "--config", f"configs/{run.model}.yaml", *ov],
                               env=env, stdout=fh, stderr=subprocess.STDOUT)
        tock(f"{run.name}:{step}")
        print(open(log).read()[-1500:])
        assert p.returncode == 0, f"{run.name} {step} failed"
    if USE_DRIVE:
        shutil.copytree(f"{WORK}/results/runs/{run.name}", f"{DRIVE_DIR}/runs/{run.name}", dirs_exist_ok=True)
        for f in glob.glob(f"{WORK}/results/{run.name}.*.log"): shutil.copy(f, f"{DRIVE_DIR}/runs/{run.name}/")

In [ ]:
# Stage E: robustness, counterfactual probes (inference only) and CUDA efficiency
if "E" in STAGES:
    for kind in ("robustness", "probes"):
        a = REG["analyses"][kind]
        for rn in a["runs"]:
            src = f"{DRIVE_DIR}/runs/{rn}" if USE_DRIVE else f"{WORK}/results/runs/{rn}"
            if not os.path.exists(f"{src}/best.pt"):
                print(f"skip {kind} for {rn}: trained run not found (run its stage first)"); continue
            if os.path.exists(f"{src}/{kind}_summary.csv"):
                print(f"skip {kind} for {rn}: done"); continue
            tick(f"{rn}:{kind}")
            p = subprocess.run([sys.executable, "-m", "bscan.robustness", "--run", src, "--mode", kind,
                                "--splits", *a["splits"], "--max_files", str(a["max_files"]), "--device", "cuda",
                                "--metadata", f"{WORK}/metadata/unified_metadata.csv", "--audio_root", f"{WORK}/data"],
                               env=env, capture_output=True, text=True)
            tock(f"{rn}:{kind}")
            print(p.stdout[-2500:]); assert p.returncode == 0, p.stderr[-3000:]
    tick("efficiency")
    p = subprocess.run([sys.executable, "-m", "bscan.efficiency", "--device", "cuda", "--rtf_files", "100",
                        "--metadata", f"{WORK}/metadata/unified_metadata.csv", "--audio_root", f"{WORK}/data",
                        "--out", f"{WORK}/results/efficiency"], env=env, capture_output=True, text=True)
    tock("efficiency"); print(p.stdout[-3000:]); assert p.returncode == 0, p.stderr[-3000:]
    if USE_DRIVE: shutil.copytree(f"{WORK}/results/efficiency", f"{DRIVE_DIR}/efficiency", dirs_exist_ok=True)

In [ ]:
# Optional: frozen XLS-R 300M + logistic regression
if RUN_OPTIONAL_SSL and not (USE_DRIVE and os.path.exists(f"{DRIVE_DIR}/runs/ssl_xlsr_probe__P1__seed42/metrics.json")):
    tick("ssl")
    p = subprocess.run([sys.executable, "-m", "bscan.ssl_probe", "--protocol", "P1", "--seed", "42",
                        "--metadata", f"{WORK}/metadata/unified_metadata.csv", "--audio_root", f"{WORK}/data",
                        "--cache", f"{WORK}/data/cache/ssl", "--out_dir", f"{WORK}/results/runs"], env=env, capture_output=True, text=True)
    tock("ssl"); print(p.stdout[-3000:]); assert p.returncode == 0, p.stderr[-3000:]
    if USE_DRIVE:
        shutil.copytree(f"{WORK}/results/runs/ssl_xlsr_probe__P1__seed42", f"{DRIVE_DIR}/runs/ssl_xlsr_probe__P1__seed42", dirs_exist_ok=True)

In [ ]:
# Progress over the whole registry (Drive)
rows = []
for r in expected_runs(REG, include_optional=True):
    if REG["models"][r.model]["kind"] == "local": continue
    rows.append({"run": r.name, "stage": r.stage, "done": on_drive(r.name) or os.path.exists(f"{WORK}/results/runs/{r.name}/metrics.json")})
prog = pd.DataFrame(rows); print(prog.groupby("stage").done.agg(["sum", "count"]))
print(prog[~prog.done].to_string(index=False) if (~prog.done).any() else "all registered Colab runs are complete")

In [ ]:
# Package everything for return
out = f"/content/BSCAN_results_{CODE_VERSION}"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
src_runs = f"{DRIVE_DIR}/runs" if USE_DRIVE else f"{WORK}/results/runs"
shutil.copytree(src_runs, f"{out}/runs", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.npy"))
for extra in ("smoke_cuda", "efficiency"):
    for base in (f"{WORK}/results", DRIVE_DIR):
        if os.path.exists(f"{base}/{extra}"): shutil.copytree(f"{base}/{extra}", f"{out}/{extra}", dirs_exist_ok=True)
json.dump({"code_version": CODE_VERSION, "stages": STAGES, "timing_sec": TIMING, "gpu": torch.cuda.get_device_name(0),
           "torch": torch.__version__, "cuda": torch.version.cuda}, open(f"{out}/colab_session.json", "w"), indent=2)
zp = shutil.make_archive(out, "zip", out)
if USE_DRIVE: shutil.copy(zp, DRIVE_DIR)
print("results:", zp, round(os.path.getsize(zp) / 2**20, 1), "MB")
from google.colab import files; files.download(zp)